---

This Notebook creation considered as base this sources

1. Sequence classification Notebook
https://colab.research.google.com/github/huggingface/notebooks/blob/main/transformers_doc/en/pytorch/sequence_classification.ipynb

2. Text Classification Notebook https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/text_classification.ipynb

3. Tasks: Text Classification guide https://huggingface.co/docs/transformers/en/tasks/sequence_classification

4. Transformers: distilbert model documentation and guide https://huggingface.co/docs/transformers/model_doc/distilbert

5. Create a weighted loss function to handle imbalance? https://discuss.huggingface.co/t/create-a-weighted-loss-function-to-handle-imbalance/138178
---

Citation of the SEAHORSE Dataset:
E. Clark, S. Rijhwani, S. Gehrmann, J. Maynez, R. Aharoni, V. Nikolaev,
T. Sellam, A. Siddhant, D. Das, and A. P. Parikh, “Seahorse: A
multilingual, multifaceted dataset for summarization evaluation,” 2023, GitHub
Website: https://github.com/google-research-datasets/seahorse , HuggingFace
page: https://huggingface.co/datasets/SEACrowd/seahorse. [Online]. Available:
https://arxiv.org/abs/2305.13194

---

# **SEAHORSE Dataset with DestilBert**

In [9]:
# Cell useful for Run all below command

# **1. Set up of environment**

#### **1.1** Install Pytorch with Nvidia CUDA to leverage GPU parallelism.

In [10]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Aleja\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


#### **1.2** Reproducibility settings.

In [11]:
# Reproducibility configuration

import os

# Deterministic cuBLAS workspace
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


import torch
import numpy as np
import random
from transformers import set_seed


seed = 42
set_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

#### **1.3** Install packages and import libraries.

In [12]:
pip install ipython-autotime transformers peft datasets evaluate ipywidgets scikit-learn hf_xet accelerate wget tokenizers accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Aleja\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [13]:
import wget
import zipfile

from datasets import load_dataset,Dataset
from datasets import ClassLabel, Value

import torch
from torch.utils.data import DataLoader

from transformers import DataCollatorWithPadding, AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer


import gc

import pandas as pd
import numpy as np

from scipy.stats import pearsonr

import time
import datetime

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

import os
from pathlib import Path
os.environ["WANDB_DISABLED"] = "true"

#### 1.4 Measuring time with CUDA

How to measure Time of execution of a cell using CUDA

```python
start = time.perf_counter()

# code using GPU

torch.cuda.synchronize()
end = time.perf_counter()
elapsed =  end - start
```

In [14]:
# method to print time spent training when obtained elapsed result using cuda
def print_training_time(elapsed):
    print("\n\n________________________________________",
          "Training Time:",
          str(datetime.timedelta(seconds=elapsed)),
          "[" + str(elapsed) + " Seconds]",
          "________________________________________\n", sep="\n")

In [15]:
# method to print time spent training when obtained elapsed result using cuda
def print_inference_time(elapsed):
    print("\n\n________________________________________",
          "Inference Time:",
          str(datetime.timedelta(seconds=elapsed)),
          "[" + str(elapsed) + " Seconds]",
          "________________________________________\n", sep="\n")

#### **1.5** Check that PyTorch uses GPU as Cuda device

In [16]:
from transformers.utils import is_torch_available, is_tf_available, is_flax_available
print("Backends for Hugging Face Transformers Library:")
print("    PyTorch:", is_torch_available())
print("    TensorFlow:", is_tf_available())
print("    JAX/Flax:", is_flax_available())
print()
print("GPU Device =",torch.cuda.is_available()," | ", torch.cuda.get_device_name(0))

Backends for Hugging Face Transformers Library:
    PyTorch: True
    TensorFlow: False
    JAX/Flax: False

GPU Device = True  |  NVIDIA GeForce GTX 1650


Print and save into a text file the environment requirements for replicability

#### **1.6** Save versions of packages and libraries to support future reproducibility.

In [17]:
pip freeze

accelerate==1.12.0
aiohappyeyeballs==2.6.1
aiohttp==3.11.18
aiosignal==1.3.2
anyio==4.9.0
argon2-cffi==23.1.0
argon2-cffi-bindings==21.2.0
arrow==1.3.0
asttokens==3.0.0
async-lru==2.0.5
attrs==25.3.0
babel==2.17.0
beautifulsoup4==4.13.4
bleach==6.2.0
certifi==2025.1.31
cffi==1.17.1
charset-normalizer==3.4.1
colorama==0.4.6
comm==0.2.2
contourpy==1.3.2
cycler==0.12.1
datasets==4.4.1
debugpy==1.8.14
decorator==5.2.1
defusedxml==0.7.1
dill==0.3.8
evaluate==0.4.6
executing==2.2.0
fastjsonschema==2.21.1
filelock==3.18.0
fonttools==4.57.0
fqdn==1.5.1
frozenlist==1.6.0
fsspec==2024.12.0
graphviz==0.21
h11==0.16.0
hf-xet==1.2.0
httpcore==1.0.9
httpx==0.28.1
huggingface-hub==0.36.0
idna==3.10
ipykernel==6.29.5
ipython==9.1.0
ipython-autotime==0.3.2
ipython_pygments_lexers==1.1.1
ipywidgets==8.1.8
isoduration==20.11.0
jedi==0.19.2
Jinja2==3.1.6
joblib==1.4.2
json5==0.12.0
jsonpointer==3.0.0
jsonschema==4.23.0
jsonschema-specifications==2025.4.1
jupyter-events==0.12.0
jupyter-lsp==2.2.5
jupyter_c

In [18]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


# **2.** Functions to compute performance metrics during training

In [19]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def compute_model_metrics(predictions_class_1, y_true, threshold_hp = 0.5, print_report=False):
    """
    :param predictions_class_1: Vector of membership probabilities for Class 1 (that instance belongs to class 1).
    :param y_true:  Vector with correct True labels for each sample in dataset.
    :param threshold_hp: A [0-1] custom [>=] threshhold to assign class 1 membership (Optional with default 0.5).
    :param print_report: If true, prints out all computed metrics including the confusion matrix values. If False, returns a dictionary with name and value of metrics. Useful for metrics post epoch evaluation during training.
    """

    assert (0.0 <= threshold_hp <= 1.0), f"Threshold_hp must be a number in range [0-1]"
    y_pred  = (predictions_class_1 >= threshold_hp).long() # Model predicted classification Class-labels
                                                           # long() casts boolean to Int64 in Tensors


    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    accuracy       = accuracy_score(y_true, y_pred)
    precision      = precision_score(y_true, y_pred)
    recall         = recall_score(y_true, y_pred)  # a.k.a. sensitivity
    specificity    = tn / (tn + fp)
    f1             = f1_score(y_true, y_pred)

    youden_j_statistic   = recall + specificity - 1.0
    fpr, tpr, thresholds = roc_curve(y_true, predictions_class_1)
    roc_auc              = auc(fpr, tpr)

    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html
    # https://en.wikipedia.org/wiki/Pearson_correlation_coefficient
    pearson_correlation_coefficient, p_value = pearsonr(y_true, predictions_class_1)

    if (print_report):
        print("TN =",tn, "FP =",fp, "FN =",fn, "TP =",tp)
        print(f"Accuracy                  : {accuracy:.4f}", " |",tn + tp, "/" ,tn+fp+fn+tp)
        print(f"Precision                 : {precision:.4f}", " |",tp, "/" ,tp+fp)
        print(f"Sensitivity (Recall, TPR) : {recall:.4f}", " |", tp, "/" ,tp+fn)
        print(f"Specificity (TNR)         : {specificity:.4f}", " |", tn, "/" ,tn+fp) #
        print(f"Youden's J statistic      : {youden_j_statistic:.4f}")
        print(f"ROC AUC                   : {roc_auc:.4f}")
        print(f"Pearson correlation       : r = {pearson_correlation_coefficient:.4f}", f" p = {p_value:.4f}")
        print(f"F1 Score                  : {f1:.4f}")
        return None
    else:
        results = {
        "Accuracy":accuracy,
        "Precision":precision,
        "Sensitivity (Recall, TPR)":recall,
        "Specificity (TNR)":specificity,
        "Youden's J statistic":youden_j_statistic,
        "ROC AUC":roc_auc,
        "Pearson correlation":pearson_correlation_coefficient,
        "F1":f1
        }
        return results

In [20]:
# Metrics to report during training after an epoch completes.
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = torch.from_numpy(predictions) # logits
    softmax_predictions_both_classes = torch.softmax(predictions, dim=1) # Softmax needed for ROC AUC and Pearson correlation
    probabilities_class_1 = softmax_predictions_both_classes[:, 1]
    return compute_model_metrics(probabilities_class_1, torch.from_numpy(labels) )

# **3.** Functions and methods for test set evaluation

In [21]:
def compute_model_predictions(X, pretrained_model_path, data_loader_batch_size = 32, custom_model_class=False):

    #1. Load pretrained model, tokenizer and data loader

    if custom_model_class == "DistilBertDualHeadClassifier":
        model_for_inference = DistilBertDualHeadClassifier.from_pretrained(pretrained_model_path)
    else:
        model_for_inference = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_path   #"model/checkpoint-123" , dtype=torch.float16
    )



    input_text_tokenizer = AutoTokenizer.from_pretrained(pretrained_model_path)

    #Collate function and dataloader for parallel inference
    def collate_input(batch):
        return input_text_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)

    # Use Dataloader class passing as input a List of strings.
    # This is Ok as List provides all required operations that a Dataset would provide for building a DataLoader
    loader = DataLoader(X, batch_size=data_loader_batch_size, collate_fn=collate_input)



    #2. Perform inference using X data
    #switch pytorch from training mode to evaluation-inference mode. Important to disable train only operations.
    model_for_inference.eval()

    #If available save model in GPU
    inference_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_for_inference.to(inference_device)


    all_outputs = []
    with torch.no_grad():
      for inputs in loader:
          inputs = {k: v.to(inference_device) for k, v in inputs.items()} #If available save
                                                                # data in GPU
          logits = model_for_inference(**inputs).logits
          all_outputs.append(logits.detach().cpu())
          del inputs, logits
          torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    inference_outputs_tensor = torch.cat(all_outputs, dim=0)


    #switch/restore train mode
    model_for_inference.train()

    return inference_outputs_tensor

In [22]:
def plot_roc_curve(y_true, probabilities):
    # Compute ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, probabilities)
    roc_auc = auc(fpr, tpr)

    # Compute Youden's J statistic
    j_scores = tpr - fpr
    j_max_index = np.argmax(j_scores)
    Youden_threshold = thresholds[j_max_index]

    #balance point
    diff = np.abs(tpr - (1 - fpr))
    balance_index = np.argmin(diff)
    balance_threshold = thresholds[balance_index]

    # Plot ROC
    plt.figure(facecolor='white')
    plt.gca().set_facecolor('white')
    plt.gca().set_aspect('equal', adjustable='box')
    plt.gca()

    plt.plot([0, 1], [0, 1], 'k--') #black dash line

    handle_curve,  = plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.6f})')

    handle_youden = plt.scatter(fpr[j_max_index], tpr[j_max_index], color='magenta',
                                label=f'max Youden J = {Youden_threshold:.6f}')

    handle_balance = plt.scatter(fpr[balance_index], tpr[balance_index], color='green',
                                 label=f'Balance = {balance_threshold:.2f}')

    index_untuned = np.where(thresholds >= 0.5)[0][-1] #thresholds lists in ROC function returned in decreasing order.
    handle_untuned = plt.scatter(fpr[index_untuned], tpr[index_untuned], color='navy',
                                 label=f'Untuned = 0.50')

    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve Thresholds', fontweight='bold')


    ax = plt.gca()
    # First legend (curves)
    legend1 = ax.legend(
        handles=[handle_untuned, handle_youden, handle_balance],
        loc="lower right",
        title="Thresholds"
    )
    ax.add_artist(legend1)   # keep legend1

    legend2 = ax.legend(
        handles=[handle_curve],
        loc="lower right",
        bbox_to_anchor=(1.0, 0.25)
    )

    plt.grid(True)

    os.makedirs("Saved_images", exist_ok=True)
    plt.savefig("Saved_images/roc_curve.png", facecolor='white', dpi=300)


    plt.show()

In [23]:
import re # regular expressions

def sort_files_numerically(file_list):
    """
    Sort a list of Strings using the numbers contained in them.
    Used to sort epochs checkpoints e.g. ['checkpoint-1', 'checkpoint-7', ''checkpoint-20']
    """
    # Extract number from the filename using regex
    def extract_number(filename):
        match = re.search(r'(\d+)', filename)
        return int(match.group(1)) if match else -1

    return sorted(file_list, key=extract_number)

In [24]:
def metrics_reload_model(input_X_data, y_true_labels, model_checkpoint_path, custom_model_class = False):
    """
    Using the folder where a model was saved, compute the metrics.
    """
    logits_predictions = compute_model_predictions(input_X_data, model_checkpoint_path, custom_model_class=custom_model_class)
    softmax_predictions = torch.softmax(logits_predictions, dim=1)  # Softmax for probabilistic interpretation.
    compute_model_metrics(softmax_predictions[:, 1], y_true_labels, print_report=True)

In [25]:
def metrics_all_epochs_post_training(input_X_data, y_true_labels, model_checkpoints_path):
    """
    Using the folder where a model was saved, compute the metrics over the epochs-checkpoints.
    """
    epoch_i = 1
    subfolders = [f for f in os.listdir(model_checkpoints_path) if os.path.isdir(os.path.join(model_checkpoints_path, f))]
    subfolders = sort_files_numerically(subfolders)

    for model_checkpoint in subfolders:

        print("\nEpoch " + str(epoch_i),"[" + model_checkpoint + "]")
        epoch_i += 1

        logits_predictions = compute_model_predictions(input_X_data, model_checkpoints_path + "/" +  model_checkpoint)
        softmax_predictions = torch.softmax(logits_predictions, dim=1) # Softmax for probabilistic interpretation.
        compute_model_metrics(softmax_predictions[:, 1], y_true_labels, print_report=True)


In [26]:
def obtain_language_separated_subdatasets(Dataset_object):
    languages = Dataset_object.unique("language")
    language_separated_datasets = {}
    for language in languages:
        filtered = Dataset_object.filter(lambda row: row["language"] == language)
        language_separated_datasets[language] = filtered
    return language_separated_datasets

In [27]:
def assess_per_language_performance(Dataset_object, model_path):
    language_separated_datasets = obtain_language_separated_subdatasets(Dataset_object)
    for language, subdataset_language_i in language_separated_datasets.items():
        logits_predictions_language_i = compute_model_predictions(subdataset_language_i["text"], model_path)
        softmax_predictions_class_1 = torch.softmax(logits_predictions_language_i, dim=1)
        print("Language:", language)
        compute_model_metrics(softmax_predictions_class_1[:, 1], subdataset_language_i["labels"])
        print()

# **4.** Load data

## **4.1** Training and validation data

We start loading the preprocessed data (See the `Data_preparation.ipynb` Notebook).

The datasets we pass to Trainer must have their target column named as `"labels"`

In [28]:
comprehensible_train_dataframe      = pd.read_csv("unbalanced_seahorse/seahorse_dataset_train_df_unbalanced.csv")
comprehensible_validation_dataframe = pd.read_csv("unbalanced_seahorse/seahorse_dataset_validation_df_unbalanced.csv")
Dataset_object_comprehensible_train      = Dataset.from_pandas(comprehensible_train_dataframe).rename_column("comprehensible", "labels")
Dataset_object_comprehensible_validation = Dataset.from_pandas(comprehensible_validation_dataframe).rename_column("comprehensible", "labels")

grammar_train_dataframe      = pd.read_csv("unbalanced_seahorse/seahorse_dataset_train_df_subset_comprehensible_is_yes_unbalanced.csv")
grammar_validation_dataframe = pd.read_csv("unbalanced_seahorse/seahorse_dataset_validation_df_subset_comprehensible_is_yes_unbalanced.csv")
Dataset_object_grammar_train      = Dataset.from_pandas(grammar_train_dataframe).rename_column("grammatical", "labels")
Dataset_object_grammar_validation = Dataset.from_pandas(grammar_validation_dataframe).rename_column("grammatical", "labels")

repetition_train_dataframe      = pd.read_csv("unbalanced_seahorse/seahorse_dataset_train_df_subset_comprehensible_is_yes_unbalanced.csv")
repetition_validation_dataframe = pd.read_csv("unbalanced_seahorse/seahorse_dataset_validation_df_subset_comprehensible_is_yes_unbalanced.csv")
Dataset_object_repetition_train      = Dataset.from_pandas(repetition_train_dataframe).rename_column("repetition_free", "labels")
Dataset_object_repetition_validation = Dataset.from_pandas(repetition_validation_dataframe).rename_column("repetition_free", "labels")

Target-Label column must be of type `Int64` for Classification models.

In [29]:
Dataset_object_comprehensible_train      = Dataset_object_comprehensible_train.cast_column("labels", Value("int64"))
Dataset_object_comprehensible_validation = Dataset_object_comprehensible_validation.cast_column("labels", Value("int64"))

Dataset_object_grammar_train      = Dataset_object_grammar_train.cast_column("labels", Value("int64"))
Dataset_object_grammar_validation = Dataset_object_grammar_validation.cast_column("labels", Value("int64"))

Dataset_object_repetition_train      = Dataset_object_repetition_train.cast_column("labels", Value("int64"))
Dataset_object_repetition_validation = Dataset_object_repetition_validation.cast_column("labels", Value("int64"))

Casting the dataset:   0%|          | 0/60470 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8895 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/54566 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8025 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/54566 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8025 [00:00<?, ? examples/s]

Apply Tokenization

In [30]:
# Tokenizer and Collator to use whenever data require processing to fit format or batch sizes.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")

#collator with padding allows to adjust dynamically each batch sample to the max batch length, for correct parallelization.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [31]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True) # In each row the name of field is text

In [32]:
tokenized_comprehensible_train = Dataset_object_comprehensible_train.map(preprocess_function, batched=True)
tokenized_comprehensible_validation = Dataset_object_comprehensible_validation.map(preprocess_function, batched=True)

tokenized_grammar_train = Dataset_object_grammar_train.map(preprocess_function, batched=True)
tokenized_grammar_validation = Dataset_object_grammar_validation.map(preprocess_function, batched=True)

tokenized_repetition_train = Dataset_object_repetition_train.map(preprocess_function, batched=True)
tokenized_repetition_validation = Dataset_object_repetition_validation.map(preprocess_function, batched=True)

Map:   0%|          | 0/60470 [00:00<?, ? examples/s]

Map:   0%|          | 0/8895 [00:00<?, ? examples/s]

Map:   0%|          | 0/54566 [00:00<?, ? examples/s]

Map:   0%|          | 0/8025 [00:00<?, ? examples/s]

Map:   0%|          | 0/54566 [00:00<?, ? examples/s]

Map:   0%|          | 0/8025 [00:00<?, ? examples/s]

## **4.2** Test data

For robust evaluation we avoided preprocessing the test data.

We perform the minimal modifications needed for test set to be run in the model:

* In SEAHORSE only rows considered comprehensible were further annotated for repetition_free and grammatical. Therefore, Apply a filter.
* Change the label format `No:0 Yes:1`.
* Remove rows WITH `Unsure` rating.
Not any preprocessing applied to test set. Tokenization will be done at inference time.
We load the Xs texts and y labels directly from the original dataset.

In [33]:
# In case we already have the dataset file, we skip download+extract.
file_path = Path("seahorse_data.zip")
if not file_path.is_file():
    url = "https://storage.googleapis.com/seahorse-public/seahorse_data.zip"
    filename = wget.download(url)
    with zipfile.ZipFile("seahorse_data.zip", 'r') as zip_ref:
        zip_ref.extractall()

In [34]:
seahorse_dataset = load_dataset(
    "csv",
    data_files = {
        "train": "seahorse_data/train.tsv",
        "validation": "seahorse_data/validation.tsv",
        "test": "seahorse_data/test.tsv"
    },
    delimiter="\t"
)

In [35]:
test_data = seahorse_dataset["test"].to_pandas()
test_data.replace({'No': 0, 'Yes': 1}, inplace=True)

X_test    = test_data["summary"]
test_comprehensible    =   test_data[~( (test_data["question1"] == "Unsure") )]
X_test_comprehensible    = test_comprehensible["summary"]
y_test_comprehensible    = test_comprehensible["question1"]

test_repetition_free     = test_data[( (test_data["question1"] == 1) & ~(test_data["question2"] == "Unsure") )]
X_test_repetition_free   = test_repetition_free["summary"]
y_test_repetition_free   = test_repetition_free["question2"]

test_grammatical     = test_data[( (test_data["question1"] == 1) & ~(test_data["question3"] == "Unsure") )]
X_test_grammatical   = test_grammatical["summary"]
y_test_grammatical   = test_grammatical["question3"]

# **5.** Weighted Loss

We use weighted error to manage Class imbalance. This preserves the diversity of majority class observations.

#### **5.1** Compute class weights using inverse frequency and then normalization.

In [36]:
from collections import Counter
print("                           0       1    ")

print("Comprehensible:")

label_counts = Counter(tokenized_comprehensible_train["labels"])
num_classes = 2
class_counts = torch.tensor([label_counts[i] for i in range(num_classes)], dtype=torch.float)

inverse_frequency_class_weights_comprehensible = 1.0 / class_counts        # inverse frequency
inverse_frequency_class_weights_comprehensible = (inverse_frequency_class_weights_comprehensible /
                                                  inverse_frequency_class_weights_comprehensible.sum())  # optional normalization

print("class counts:  ",  class_counts)
print("class weights: ", inverse_frequency_class_weights_comprehensible)


print("\nRepetition_free:")

label_counts = Counter(tokenized_repetition_train["labels"])
num_classes = 2
class_counts = torch.tensor([label_counts[i] for i in range(num_classes)], dtype=torch.float)

inverse_frequency_class_weights_repetition = 1.0 / class_counts        # inverse frequency
inverse_frequency_class_weights_repetition = (inverse_frequency_class_weights_repetition /
                                              inverse_frequency_class_weights_repetition.sum())  # optional normalization

print("class counts:  ",  class_counts)
print("class weights: ", inverse_frequency_class_weights_repetition)



print("\nGrammatical:")

label_counts = Counter(tokenized_grammar_train["labels"])
num_classes = 2
class_counts = torch.tensor([label_counts[i] for i in range(num_classes)], dtype=torch.float)

inverse_frequency_class_weights_grammar = 1.0 / class_counts        # inverse frequency
inverse_frequency_class_weights_grammar = (inverse_frequency_class_weights_grammar /
                                              inverse_frequency_class_weights_grammar.sum())  # optional normalization

print("class counts:  ",  class_counts)
print("class weights: ", inverse_frequency_class_weights_grammar)

                           0       1    
Comprehensible:
class counts:   tensor([ 5904., 54566.])
class weights:  tensor([0.9024, 0.0976])

Repetition_free:
class counts:   tensor([ 6124., 48442.])
class weights:  tensor([0.8878, 0.1122])

Grammatical:
class counts:   tensor([ 5557., 49009.])
class weights:  tensor([0.8982, 0.1018])


#### **5.2** Extend the Trainer class to use weighted Cross Entropy Loss

We pass all basic trainer parameters, and now also the weights.

```python
trainer = WeightedCrossEntropyTrainer(
    ...
    class_weights=class_weights
)
```

In [37]:
from transformers import Trainer
import torch
import torch.nn as nn

class WeightedCrossEntropyTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights  # tensor, shape [2]

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")  # integer labels: 0 or 1
        outputs = model(**inputs)
        logits = outputs.logits  # shape [batch, 2]

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fn(logits.view(-1, 2), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


# **6.** Methods for freezing layers

In [38]:
def show_trainable_and_frozen(model):
    for name, param in model.named_parameters():
        if param.requires_grad:
            print("Trainable:", name)
        else:
            print("Frozen:", name)

In [39]:
def freeze_first_n_layers(model, n):
    for param in model.distilbert.embeddings.parameters():
        param.requires_grad = False
    layers = model.distilbert.transformer.layer
    for layer in layers[:n]:
        for param in layer.parameters():
            param.requires_grad = False

In [40]:
# Freeze all backbone parameters
def freeze_all_destilbert(model):
    for param in model.distilbert.parameters():
        param.requires_grad = False

holi

# Custom Fusion classifiers

##### DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

In [41]:
from transformers.modeling_outputs import SequenceClassifierOutput
import torch.nn as nn
from transformers import DistilBertModel, DistilBertPreTrainedModel
from torch.nn import GELU

class DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(DistilBertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.distilbert = DistilBertModel(config)


        self.cls_head = nn.Sequential(
            nn.Linear(config.dim, 768),
            nn.GELU(),
            nn.Dropout(config.seq_classif_dropout),
        )

        self.avg_head = nn.Sequential(
            nn.Linear(config.dim, 768),
            nn.GELU(),
            nn.Dropout(config.seq_classif_dropout),
        )

        self.merge_classifier = nn.Linear(768, config.num_labels)

        self.post_init()

    def avg_pooling(self, hidden_states, attention_mask):
        mask = attention_mask.clone()
        mask[:, 0] = 0  # remove CLS

        mask_expanded = mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

        return sum_embeddings / sum_mask



    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs
    ):
        bert_kwargs = {k: v for k, v in kwargs.items() if k in ["output_hidden_states", "output_attentions"]}

        return_dict = kwargs.pop("return_dict", True)
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=return_dict,
            **bert_kwargs
        )

        hidden_states = outputs.last_hidden_state  # (batch_size, seq_len, 768)

        # Branch 1: [CLS] token
        cls_output = hidden_states[:, 0, :]         # for all batch take [CLS] complete.
        cls_branch = self.cls_head(cls_output)

        # Branch 2: Average pooling
        avg_output = self.avg_pooling(hidden_states, attention_mask)
        avg_branch = self.avg_head(avg_output)

        # Merge and classify
        #merged = torch.cat([cls_branch, avg_branch], dim=1)
        merged = cls_branch + avg_branch   # (B, 768)
        logits = self.merge_classifier(merged)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states = None, # =outputs.hidden_states,
            attentions    = None, # outputs.attentions,
                )



```
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)
````

##### DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

In [42]:
from transformers.modeling_outputs import SequenceClassifierOutput
import torch.nn as nn
from transformers import DistilBertModel, DistilBertPreTrainedModel
from torch.nn import GELU

class DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(DistilBertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.distilbert = DistilBertModel(config)

        self.merge_classifier = nn.Sequential(
            nn.Linear(1536, 768),
            nn.GELU(),
            nn.Dropout(config.seq_classif_dropout),
            nn.Linear(768, 2)
        )

        self.post_init()

    def avg_pooling(self, hidden_states, attention_mask):
        mask = attention_mask.clone()
        mask[:, 0] = 0  # remove CLS

        mask_expanded = mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)

        return sum_embeddings / sum_mask



    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs
    ):
        bert_kwargs = {k: v for k, v in kwargs.items() if k in ["output_hidden_states", "output_attentions"]}

        return_dict = kwargs.pop("return_dict", True)
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=return_dict,
            **bert_kwargs
        )

        hidden_states = outputs.last_hidden_state  # (batch_size, seq_len, 768)

        # Branch 1: [CLS] token
        cls_output = hidden_states[:, 0, :]         # for all batch take [CLS] complete.
        # Branch 2: Average pooling
        avg_output = self.avg_pooling(hidden_states, attention_mask)

        # Merge and classify
        merged = torch.cat([cls_output, avg_output], dim=1)
        logits = self.merge_classifier(merged)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states = None, # outputs.hidden_states,
            attentions    = None, # outputs.attentions,
                )



```
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)
```

##### Feature extraction as CLS and average pool soncatenations of last 3 layers

In [43]:
from transformers import DistilBertModel, DistilBertPreTrainedModel, DistilBertConfig
from transformers.modeling_outputs import SequenceClassifierOutput
import torch
import torch.nn as nn
from torch.nn import GELU

class DistilBertLast3LayerFusion(DistilBertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.distilbert = DistilBertModel(config)

        # Hidden dimension for DistilBERT
        hidden_dim = config.dim
        # Last 3 layers concatenation → 3 * hidden_dim
        self.feature_dim = hidden_dim * 3  # per branch (CLS or mean)

        # Fusion head: input = CLS + Mean pooling concatenated → 2*feature_dim
        self.merge_classifier = nn.Sequential(
            nn.Linear(self.feature_dim * 2, 1024),  # possible to set smaller/larger
            GELU(),
            nn.Dropout(config.seq_classif_dropout),
            nn.Linear(1024, config.num_labels)
        )

        self.post_init()

    # Safe mean pooling ignoring CLS token
    def mean_pooling(self, hidden_states, attention_mask):
        mask = attention_mask.clone()
        mask[:, 0] = 0  # ignore CLS

        mask_exp = mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_emb = torch.sum(hidden_states * mask_exp, dim=1)
        sum_mask = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
        return sum_emb / sum_mask

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        # Safe handling of return_dict to avoid multiple keyword error
        return_dict = kwargs.pop("return_dict", True)

        # Extract hidden states and attentions keys from kwargs
        bert_kwargs = {k: v for k, v in kwargs.items() if k in ["output_hidden_states", "output_attentions"]}

        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=return_dict,
            **bert_kwargs
        )

        hidden_states = outputs.hidden_states  # tuple: (embeddings, layer1, ..., layer6)
        # Grab last 3 layers
        last_3 = hidden_states[-3:]  # tuple of 3 tensors

        # ---- Branch 1: CLS token concatenation ----
        cls_features = torch.cat([layer[:, 0, :] for layer in last_3], dim=1)  # (batch, 3*hidden_dim)

        # ---- Branch 2: Mean pooling of last 3 layers ----
        mean_features = torch.cat([self.mean_pooling(layer, attention_mask) for layer in last_3], dim=1)  # (batch, 3*hidden_dim)

        # ---- Merge both branches ----
        fusion_features = torch.cat([cls_features, mean_features], dim=1)  # (batch, 2*feature_dim)

        logits = self.merge_classifier(fusion_features)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states = None, # hidden_states=outputs.hidden_states,
            attentions    = None # attentions=outputs.attentions
        )

# **Bookmark: Run all needed before training models**

In [44]:
# This cell will appear as executed

In [37]:
# In this cell do Run all above

# 1. Initial  Exploration

# Fine Tuning of Last 3 layers (one hidden layer for [CLS] representation)

#### HuggingFace DistilBertForSequenceClassification Model


Now we createas baseline a basic DistilBert Classifier as shown in the referenced materials (see start of notebook).

HuggingFace provides the function `AutoModelForSequenceClassification`. When it is applied to _**distilbert-base-multilingual-cased**_ for a Binary/2-Class Classification problem, it creates a `DistilBertForSequenceClassification` model.

The Classification Head of this model is as follows:

*768 Units with ReLU Activation and Dropout(p=0.2) `[input features = 768, units/output features = 768]`
*2 units with Linear output `[input features = 768, units/output features = 2]`

Note also that this model will automatically use CrossEntropyLoss() as Loss function.


The source code of this HuggingFace model can be found at:

https://github.com/huggingface/transformers/blob/v4.57.3/src/transformers/models/distilbert/modeling_distilbert.py#L851

To correctly understand the architecture is important to check both the named-attribute layers and the forward pass function to see the activation functions. Note that here if we use the print function for the model we don't see the ReLu activation applied.


## Comprehensible_DBSC_FT_3_layers

In [38]:
Comprehensible_DBSC_FT_3_layers = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Comprehensible_DBSC_FT_3_layers) # See notes above on why ReLU activation does not show.

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)

Keep only last 3 layers trainable

In [40]:
freeze_first_n_layers(Comprehensible_DBSC_FT_3_layers,3)

In [42]:
show_trainable_and_frozen(Comprehensible_DBSC_FT_3_layers)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [43]:
training_args_Comprehensible_DBSC_FT_3_layers = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_DBSC_FT_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DBSC_FT_3_layers_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DBSC_FT_3_layers,
    args              = training_args_Comprehensible_DBSC_FT_3_layers,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [44]:
training_args_Comprehensible_DBSC_FT_3_layers

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DBSC_FT_3_layers.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DBSC_FT_3_layers_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.474100,0.449303,0.855312,0.966233,0.870031,0.719540,0.589571,0.879292,0.511261,0.915612
2,0.402000,0.415383,0.838786,0.970315,0.847227,0.760920,0.608147,0.891833,0.512972,0.904604
3,0.351800,0.535204,0.889713,0.958594,0.917383,0.634483,0.551866,0.877811,0.539110,0.937536
4,0.313100,0.582024,0.887690,0.958496,0.915140,0.634483,0.549623,0.876388,0.525314,0.936317




________________________________________
Training Time:
6:12:21.138774
[22341.138774199877 Seconds]
________________________________________



## Repetition_free_DBSC_FT_3_layers

In [38]:
Repetition_free_DBSC_FT_3_layers = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Repetition_free_DBSC_FT_3_layers) # See notes above on why ReLU activation does not show.

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)

Keep only last 3 layers trainable

In [40]:
freeze_first_n_layers(Repetition_free_DBSC_FT_3_layers,3)

In [41]:
show_trainable_and_frozen(Repetition_free_DBSC_FT_3_layers)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Repetition_free_DBSC_FT_3_layers = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_free_DBSC_FT_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_free_DBSC_FT_3_layers_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_free_DBSC_FT_3_layers,
    args              = training_args_Repetition_free_DBSC_FT_3_layers,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_free_DBSC_FT_3_layers.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_free_DBSC_FT_3_layers_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.290900,0.268077,0.951900,0.979983,0.965352,0.848649,0.814001,0.965735,0.805259,0.972612
2,0.245400,0.241151,0.942804,0.982841,0.951972,0.872432,0.824404,0.967125,0.789181,0.967160
3,0.199700,0.296144,0.950530,0.981468,0.962254,0.860541,0.822794,0.964173,0.798656,0.971766
4,0.155800,0.384720,0.956012,0.978850,0.971268,0.838919,0.810187,0.958040,0.809126,0.975044




________________________________________
Training Time:
5:01:27.881940
[18087.88194030011 Seconds]
________________________________________



## Grammatical_DBSC_FT_3_layers

In [38]:
Grammatical_DBSC_FT_3_layers = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Grammatical_DBSC_FT_3_layers) # See notes above on why ReLU activation does not show.

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)

Keep only last 3 layers trainable

In [40]:
freeze_first_n_layers(Grammatical_DBSC_FT_3_layers,3)

In [41]:
show_trainable_and_frozen(Grammatical_DBSC_FT_3_layers)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Grammatical_DBSC_FT_3_layers = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_DBSC_FT_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DBSC_FT_3_layers_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DBSC_FT_3_layers,
    args              = training_args_Grammatical_DBSC_FT_3_layers,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Grammatical_DBSC_FT_3_layers

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DBSC_FT_3_layers.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DBSC_FT_3_layers_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.598400,0.578200,0.828162,0.944707,0.859241,0.550186,0.409427,0.773320,0.332828,0.899949
2,0.555000,0.553005,0.758006,0.954984,0.767110,0.676580,0.443690,0.791060,0.345272,0.850799
3,0.496500,0.615944,0.834891,0.943951,0.867969,0.539033,0.407002,0.790395,0.365502,0.904367
4,0.431500,0.677099,0.835888,0.942420,0.870740,0.524164,0.394903,0.778942,0.354133,0.905163




________________________________________
Training Time:
5:00:11.172091
[18011.172091499902 Seconds]
________________________________________



# Feature extraction (last 3 layer embeddings)

## Comprehensible

In [38]:
Comprehensible_DistilBertLast3LayerFusion_config = DistilBertConfig.from_pretrained(
                                                    "distilbert-base-multilingual-cased",
                                                    num_labels=2,
                                                    dropout=0.1,
                                                    attention_dropout=0.1,
                                                    seq_classif_dropout=0.1,
                                                )

Comprehensible_DistilBertLast3LayerFusion_model = DistilBertLast3LayerFusion.from_pretrained(
    "distilbert-base-multilingual-cased",
    config=Comprehensible_DistilBertLast3LayerFusion_config,
    ignore_mismatched_sizes=True  # important for custom head
)


Some weights of DistilBertLast3LayerFusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Comprehensible_DistilBertLast3LayerFusion_model) # See notes above on why ReLU activation does not show.

DistilBertLast3LayerFusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

Freeze all DestilBERT. We use only as Feature Extractor.

In [40]:
freeze_all_destilbert(Comprehensible_DistilBertLast3LayerFusion_model)

In [41]:
show_trainable_and_frozen(Comprehensible_DistilBertLast3LayerFusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Comprehensible_DistilBertLast3LayerFusion_model = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_DistilBertLast3LayerFusion_config",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBertLast3LayerFusion_model_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBertLast3LayerFusion_model,
    args              = training_args_Comprehensible_DistilBertLast3LayerFusion_model,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Comprehensible_DistilBertLast3LayerFusion_model

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBertLast3LayerFusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBertLast3LayerFusion_model_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.498700,0.481839,0.776616,0.971719,0.774953,0.791954,0.566907,0.856867,0.445100,0.862253
2,0.441000,0.445451,0.824171,0.968528,0.832150,0.750575,0.582724,0.874415,0.486410,0.895174
3,0.434300,0.451979,0.861046,0.964300,0.878505,0.700000,0.578505,0.877609,0.516024,0.919405
4,0.408400,0.437742,0.848117,0.967236,0.860810,0.731034,0.591844,0.880155,0.510129,0.910925




________________________________________
Training Time:
1:25:12.768002
[5112.768001599936 Seconds]
________________________________________



## Repetition_free

In [37]:
Repetition_free_DistilBertLast3LayerFusion_config = DistilBertConfig.from_pretrained(
                                                    "distilbert-base-multilingual-cased",
                                                    num_labels=2,
                                                    dropout=0.1,
                                                    attention_dropout=0.1,
                                                    seq_classif_dropout=0.1,
                                                )

Repetition_free_DistilBertLast3LayerFusion_model = DistilBertLast3LayerFusion.from_pretrained(
    "distilbert-base-multilingual-cased",
    config=Repetition_free_DistilBertLast3LayerFusion_config,
    ignore_mismatched_sizes=True  # important for custom head
)


Some weights of DistilBertLast3LayerFusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_free_DistilBertLast3LayerFusion_model) # See notes above on why ReLU activation does not show.

DistilBertLast3LayerFusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

Freeze all DestilBERT. We use only as Feature Extractor.

In [39]:
freeze_all_destilbert(Repetition_free_DistilBertLast3LayerFusion_model)

In [40]:
show_trainable_and_frozen(Repetition_free_DistilBertLast3LayerFusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_free_DistilBertLast3LayerFusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_free_DistilBertLast3LayerFusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_free_DistilBertLast3LayerFusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_free_DistilBertLast3LayerFusion_model,
    args              = training_args_Repetition_free_DistilBertLast3LayerFusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_free_DistilBertLast3LayerFusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_free_DistilBertLast3LayerFusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_free_DistilBertLast3LayerFusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.348200,0.300649,0.913520,0.976070,0.924930,0.825946,0.750876,0.944987,0.700226,0.949812
2,0.306900,0.291844,0.915389,0.978822,0.924366,0.846486,0.770853,0.950025,0.711762,0.950815
3,0.295600,0.282484,0.897819,0.981447,0.901549,0.869189,0.770738,0.954262,0.693891,0.939803
4,0.281200,0.280482,0.903676,0.980702,0.909014,0.862703,0.771717,0.954781,0.702853,0.943498




________________________________________
Training Time:
0:59:56.396296
[3596.3962959998753 Seconds]
________________________________________



## Grammatical

In [38]:
Grammatical_DistilBertLast3LayerFusion_config = DistilBertConfig.from_pretrained(
                                                    "distilbert-base-multilingual-cased",
                                                    num_labels=2,
                                                    dropout=0.1,
                                                    attention_dropout=0.1,
                                                    seq_classif_dropout=0.1,
                                                )

Grammatical_DistilBertLast3LayerFusion_model = DistilBertLast3LayerFusion.from_pretrained(
    "distilbert-base-multilingual-cased",
    config=Grammatical_DistilBertLast3LayerFusion_config,
    ignore_mismatched_sizes=True  # important for custom head
)


Some weights of DistilBertLast3LayerFusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Grammatical_DistilBertLast3LayerFusion_model) # See notes above on why ReLU activation does not show.

DistilBertLast3LayerFusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

Freeze all DestilBERT. We use only as Feature Extractor.

In [40]:
freeze_all_destilbert(Grammatical_DistilBertLast3LayerFusion_model)

In [41]:
show_trainable_and_frozen(Grammatical_DistilBertLast3LayerFusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Grammatical_DistilBertLast3LayerFusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_DistilBertLast3LayerFusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBertLast3LayerFusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBertLast3LayerFusion_model,
    args              = training_args_Grammatical_DistilBertLast3LayerFusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Grammatical_DistilBertLast3LayerFusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBertLast3LayerFusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBertLast3LayerFusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.618400,0.602309,0.775701,0.942358,0.799529,0.562577,0.362106,0.737787,0.292473,0.865088
2,0.596300,0.591319,0.737944,0.947820,0.749931,0.630731,0.380662,0.749624,0.293823,0.837342
3,0.573100,0.593180,0.781807,0.944978,0.804239,0.581165,0.385404,0.753933,0.309372,0.868947
4,0.564600,0.589364,0.792523,0.942752,0.819063,0.555143,0.374206,0.758198,0.315123,0.876566




________________________________________
Training Time:
0:57:18.739074
[3438.7390743000433 Seconds]
________________________________________



# Sum Fusion (2 layers)

## Comprehensible

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [40]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.478700,0.438797,0.816414,0.971526,0.820561,0.778161,0.598722,0.876221,0.486834,0.889685
2,0.410200,0.412627,0.841372,0.971217,0.849346,0.767816,0.617162,0.893181,0.517096,0.906202
3,0.383300,0.462836,0.872625,0.963920,0.892212,0.691954,0.584166,0.889048,0.536085,0.926681
4,0.338700,0.469338,0.870152,0.965700,0.887601,0.709195,0.596797,0.892284,0.532740,0.925005




________________________________________
Training Time:
2:00:54.307252
[7254.307251500199 Seconds]
________________________________________



## Repetition

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [40]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.298900,0.258893,0.945047,0.981490,0.955915,0.861622,0.817537,0.965142,0.790786,0.968534
2,0.247600,0.247562,0.943427,0.982013,0.953521,0.865946,0.819467,0.965721,0.788698,0.967558
3,0.216000,0.274836,0.942181,0.982548,0.951549,0.870270,0.821820,0.964173,0.782113,0.966800
4,0.193900,0.294793,0.947165,0.981813,0.958028,0.863784,0.821812,0.962250,0.789183,0.969775




________________________________________
Training Time:
1:21:23.758586
[4883.7585862998385 Seconds]
________________________________________



## Grammatical

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [40]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.602200,0.589330,0.816324,0.940897,0.849127,0.522924,0.372052,0.759668,0.324854,0.892659
2,0.563500,0.559085,0.757508,0.955110,0.766417,0.677819,0.444236,0.785188,0.335731,0.850423
3,0.516500,0.601657,0.825171,0.944232,0.856193,0.547708,0.403900,0.786017,0.357828,0.898060
4,0.475700,0.606159,0.814704,0.945161,0.842893,0.562577,0.405470,0.782092,0.349396,0.891102




________________________________________
Training Time:
1:21:42.271721
[4902.271721499972 Seconds]
________________________________________



# Concat Fusion (2 layers)

## Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [40]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [43]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [44]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.474700,0.449394,0.819674,0.969990,0.825670,0.764368,0.590038,0.874103,0.484506,0.892030
2,0.409100,0.410803,0.830467,0.972178,0.836012,0.779310,0.615323,0.893640,0.511287,0.898968
3,0.370800,0.475154,0.869477,0.965167,0.887352,0.704598,0.591950,0.889116,0.535220,0.924625
4,0.332200,0.479402,0.871276,0.965242,0.889346,0.704598,0.593943,0.892094,0.535608,0.925741




________________________________________
Training Time:
2:01:38.080620
[7298.080620199908 Seconds]
________________________________________



## Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [40]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.294700,0.252046,0.941308,0.981829,0.951268,0.864865,0.816132,0.964413,0.781179,0.966307
2,0.248100,0.242637,0.942181,0.983110,0.950986,0.874595,0.825581,0.964464,0.783736,0.966781
3,0.212800,0.271095,0.941558,0.982816,0.950563,0.872432,0.822996,0.963502,0.777234,0.966421
4,0.185000,0.313027,0.947913,0.978379,0.962394,0.836757,0.799151,0.959550,0.790273,0.970321




________________________________________
Training Time:
1:28:56.401461
[5336.401460899971 Seconds]
________________________________________



## Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [40]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.599000,0.587982,0.790405,0.942173,0.817124,0.551425,0.368549,0.757403,0.316953,0.875204
2,0.564400,0.563792,0.753645,0.954396,0.762538,0.674102,0.436640,0.782021,0.330570,0.847747
3,0.506100,0.618866,0.820685,0.945292,0.849820,0.560099,0.409919,0.785987,0.357227,0.895017
4,0.459700,0.642702,0.824424,0.945544,0.853976,0.560099,0.414075,0.778442,0.348669,0.897430




________________________________________
Training Time:
1:27:35.204760
[5255.2047604 Seconds]
________________________________________



# Low Rank Adaption (LoRA)

## LoRA (Comprehensible)

In [37]:
Comprehensible_FT_for_LoRA = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
from peft import LoraConfig, TaskType, get_peft_model

# Instantiate in same cell model for safety. A warning appears when we apply LoRA several times to a same object.

Comprehensible_FT_for_LoRA_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    # Last 3 layers: 3, 4, 5
    target_modules = r".*layer\.(3|4|5)\.attention\.(q_lin|k_lin|v_lin|out_lin)",
    bias="none",
)

Comprehensible_FT_LoRA_model = get_peft_model(Comprehensible_FT_for_LoRA, Comprehensible_FT_for_LoRA_config)
Comprehensible_FT_LoRA_model.print_trainable_parameters()



trainable params: 739,586 || all params: 136,065,796 || trainable%: 0.5436


Inspect architecture

In [39]:
print(Comprehensible_FT_LoRA_model) # See notes above on why ReLU activation does not show.

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(119547, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-2): 3 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): Linear(in_features=768, out_features=768, bias=True)
                (k_lin): Linear(in_features=768, out_features=768, bias=True)
                (v_lin): Linear(in_features=768, out_features=768, bias=True)
                (out_lin): Linear(in_features=768, out_features=768, bias=True)
              )
           

See frozen base model

In [40]:
show_trainable_and_frozen(Comprehensible_FT_LoRA_model)

Frozen: base_model.model.distilbert.embeddings.word_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.position_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: base_model.model.

Train the model

In [41]:
training_args_Comprehensible_FT_LoRA_model = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_FT_LoRA_model",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_FT_LoRA_model_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_FT_LoRA_model,
    args              = training_args_Comprehensible_FT_LoRA_model,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_FT_LoRA_model

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
import torch

# Force out-of-memory errors instead of spilling
#torch.cuda.set_per_process_memory_fraction(0.95, device=0)

# Disable memory growth that might cause spills
#os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:False'

# Check if you're spilling to CPU
#print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


# If max_memory > 4GB, you're likely spilling or will crash

In [44]:
import torch

# Get device name
print(f"Device name: {torch.cuda.get_device_name(0)}")

# Current memory usage
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Total: {total:.2f} GB")
print(f"Free: {total - reserved:.2f} GB")
print(f"Max GPU Memory: {torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB") # peak usage

Device name: NVIDIA GeForce GTX 1650
Allocated: 0.51 GB
Reserved: 0.54 GB
Total: 4.00 GB
Free: 3.46 GB
Max GPU Memory: 0.51 GB


In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_FT_LoRA_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_FT_LoRA_model_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.490100,0.464187,0.848567,0.963107,0.865296,0.694253,0.559549,0.862677,0.486511,0.911585
2,0.439500,0.443357,0.828331,0.968561,0.836885,0.749425,0.586310,0.873660,0.489629,0.897921
3,0.436800,0.456884,0.845082,0.965802,0.858692,0.719540,0.578232,0.875501,0.503762,0.909103
4,0.418500,0.448897,0.840247,0.967044,0.851963,0.732184,0.584147,0.877124,0.502349,0.905863




________________________________________
Training Time:
3:50:01.111652
[13801.111652100002 Seconds]
________________________________________



## LoRA (Repetition)

In [37]:
Repetition_FT_for_LoRA = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
from peft import LoraConfig, TaskType, get_peft_model

# Instantiate in same cell model for safety. A warning appears when we apply LoRA several times to a same object.

Repetition_FT_for_LoRA_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    # Last 3 layers: 3, 4, 5
    target_modules = r".*layer\.(3|4|5)\.attention\.(q_lin|k_lin|v_lin|out_lin)",
    bias="none",
)

Repetition_FT_LoRA_model = get_peft_model(Repetition_FT_for_LoRA, Repetition_FT_for_LoRA_config)
Repetition_FT_LoRA_model.print_trainable_parameters()



trainable params: 739,586 || all params: 136,065,796 || trainable%: 0.5436


Inspect architecture

In [39]:
print(Repetition_FT_LoRA_model) # See notes above on why ReLU activation does not show.

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(119547, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-2): 3 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): Linear(in_features=768, out_features=768, bias=True)
                (k_lin): Linear(in_features=768, out_features=768, bias=True)
                (v_lin): Linear(in_features=768, out_features=768, bias=True)
                (out_lin): Linear(in_features=768, out_features=768, bias=True)
              )
           

See frozen base model

In [40]:
show_trainable_and_frozen(Repetition_FT_LoRA_model)

Frozen: base_model.model.distilbert.embeddings.word_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.position_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: base_model.model.

Train the model

In [41]:
training_args_Repetition_FT_LoRA_model = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_FT_LoRA_model",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_FT_LoRA_model_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_FT_LoRA_model,
    args              = training_args_Repetition_FT_LoRA_model,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_FT_LoRA_model

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
import torch

# Force out-of-memory errors instead of spilling
#torch.cuda.set_per_process_memory_fraction(0.95, device=0)

# Disable memory growth that might cause spills
#os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:False'

# Check if you're spilling to CPU
#print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


# If max_memory > 4GB, you're likely spilling or will crash

In [44]:
import torch

# Get device name
print(f"Device name: {torch.cuda.get_device_name(0)}")

# Current memory usage
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Total: {total:.2f} GB")
print(f"Free: {total - reserved:.2f} GB")
print(f"Max GPU Memory: {torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB") # peak usage

Device name: NVIDIA GeForce GTX 1650
Allocated: 0.51 GB
Reserved: 0.54 GB
Total: 4.00 GB
Free: 3.46 GB
Max GPU Memory: 0.51 GB


In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_FT_LoRA_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_FT_LoRA_model_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.307100,0.270369,0.925483,0.980633,0.934225,0.858378,0.792604,0.956255,0.745320,0.956867
2,0.270200,0.251009,0.942056,0.981565,0.952394,0.862703,0.815097,0.962851,0.782709,0.966760
3,0.264100,0.248785,0.932710,0.983206,0.940000,0.876757,0.816757,0.964847,0.763701,0.961118
4,0.257300,0.251182,0.937695,0.982738,0.946197,0.872432,0.818630,0.965574,0.773904,0.964122




________________________________________
Training Time:
3:28:46.480848
[12526.4808481 Seconds]
________________________________________



## LoRA (Grammatical)

In [37]:
Grammatical_FT_for_LoRA = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [39]:
from peft import LoraConfig, TaskType, get_peft_model

# Instantiate in same cell model for safety. A warning appears when we apply LoRA several times to a same object.

Grammatical_FT_for_LoRA_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    # Last 3 layers: 3, 4, 5
    target_modules = r".*layer\.(3|4|5)\.attention\.(q_lin|k_lin|v_lin|out_lin)",
    bias="none",
)

Grammatical_FT_LoRA_model = get_peft_model(Grammatical_FT_for_LoRA, Grammatical_FT_for_LoRA_config)
Grammatical_FT_LoRA_model.print_trainable_parameters()



trainable params: 739,586 || all params: 136,065,796 || trainable%: 0.5436


Inspect architecture

In [40]:
print(Grammatical_FT_LoRA_model) # See notes above on why ReLU activation does not show.

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(119547, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-2): 3 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): Linear(in_features=768, out_features=768, bias=True)
                (k_lin): Linear(in_features=768, out_features=768, bias=True)
                (v_lin): Linear(in_features=768, out_features=768, bias=True)
                (out_lin): Linear(in_features=768, out_features=768, bias=True)
              )
           

See frozen base model

In [41]:
show_trainable_and_frozen(Grammatical_FT_LoRA_model)

Frozen: base_model.model.distilbert.embeddings.word_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.position_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: base_model.model.

Train the model

In [46]:
training_args_Grammatical_FT_LoRA = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_FT_LoRA_model",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_FT_LoRA_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_FT_LoRA_model,
    args              = training_args_Grammatical_FT_LoRA,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [47]:
training_args_Grammatical_FT_LoRA

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [48]:
import torch

# Force out-of-memory errors instead of spilling
#torch.cuda.set_per_process_memory_fraction(0.95, device=0)

# Disable memory growth that might cause spills
#os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:False'

# Check if you're spilling to CPU
#print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


# If max_memory > 4GB, you're likely spilling or will crash

In [49]:
import torch

# Get device name
print(f"Device name: {torch.cuda.get_device_name(0)}")

# Current memory usage
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Total: {total:.2f} GB")
print(f"Free: {total - reserved:.2f} GB")
print(f"Max GPU Memory: {torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB") # peak usage

Device name: NVIDIA GeForce GTX 1650
Allocated: 0.51 GB
Reserved: 0.54 GB
Total: 4.00 GB
Free: 3.46 GB
Max GPU Memory: 0.51 GB


In [50]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_FT_LoRA_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_FT_LoRA_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.605700,0.604695,0.781184,0.940058,0.808257,0.539033,0.347291,0.732505,0.292171,0.869190
2,0.591400,0.595882,0.754517,0.944444,0.772513,0.593556,0.366070,0.743993,0.292656,0.849870
3,0.572700,0.599109,0.790405,0.940344,0.818925,0.535316,0.354241,0.748399,0.306089,0.875444
4,0.571200,0.598121,0.801495,0.939522,0.832918,0.520446,0.353364,0.751581,0.310772,0.883014




________________________________________
Training Time:
3:25:46.534889
[12346.534888899994 Seconds]
________________________________________



# Fusions (3 layers)

## Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion (3 Layers)

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 3 layers

In [40]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 3)

In [41]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.470500,0.434517,0.811355,0.973869,0.812710,0.798851,0.611561,0.882185,0.486731,0.886021
2,0.398300,0.417467,0.853850,0.969295,0.865421,0.747126,0.612547,0.891907,0.525362,0.914417
3,0.356800,0.537905,0.884992,0.959932,0.910530,0.649425,0.559955,0.880697,0.537811,0.934578
4,0.300100,0.567634,0.886228,0.958426,0.913520,0.634483,0.548003,0.881704,0.530909,0.935434




________________________________________
Training Time:
6:28:20.450231
[23300.450230899994 Seconds]
________________________________________



## Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion (3 layers)

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 3 layers

In [40]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 3)

In [41]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.469500,0.442328,0.832265,0.970609,0.839502,0.765517,0.605019,0.880182,0.496921,0.900307
2,0.402600,0.419982,0.842383,0.969383,0.852212,0.751724,0.603936,0.890516,0.515109,0.907029
3,0.354000,0.518539,0.881057,0.961574,0.904299,0.666667,0.570966,0.881919,0.536618,0.932058
4,0.311200,0.579652,0.884205,0.958689,0.910903,0.637931,0.548834,0.879532,0.528074,0.934185




________________________________________
Training Time:
8:23:53.232683
[30233.232682700007 Seconds]
________________________________________



## Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion (3 layers)

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [40]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 3)

In [41]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [43]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [44]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.288000,0.282505,0.953396,0.979196,0.967887,0.842162,0.810049,0.964928,0.811589,0.973509
2,0.239000,0.243268,0.947040,0.982507,0.957183,0.869189,0.826372,0.965663,0.799804,0.969680
3,0.190100,0.325191,0.954019,0.980992,0.966761,0.856216,0.822977,0.962144,0.804569,0.973824
4,0.143100,0.393087,0.956885,0.979007,0.972113,0.840000,0.812113,0.958508,0.806403,0.975548




________________________________________
Training Time:
6:46:37.109870
[24397.109870199987 Seconds]
________________________________________



## Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion (3 layers)

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 3 layers

In [40]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 3)

In [41]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.290300,0.254334,0.945047,0.981629,0.955775,0.862703,0.818477,0.966791,0.793598,0.968529
2,0.241100,0.235431,0.939564,0.985184,0.945915,0.890811,0.836726,0.966972,0.782302,0.965151
3,0.196100,0.283649,0.949159,0.982689,0.959437,0.870270,0.829707,0.965210,0.796843,0.970924
4,0.148200,0.365609,0.955763,0.978165,0.971690,0.833514,0.805204,0.957194,0.807556,0.974917




________________________________________
Training Time:
5:14:00.965649
[18840.96564909999 Seconds]
________________________________________



## Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion (3 layers)

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [40]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 3)

In [41]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.592100,0.572331,0.821931,0.944829,0.851759,0.555143,0.406902,0.780309,0.341036,0.895883
2,0.551500,0.556611,0.755639,0.953894,0.765309,0.669145,0.434454,0.787096,0.343364,0.849258
3,0.487600,0.624564,0.842118,0.941665,0.878914,0.513011,0.391925,0.786603,0.364698,0.909208
4,0.411500,0.708912,0.835389,0.940798,0.871848,0.509294,0.381142,0.765224,0.344620,0.905012




________________________________________
Training Time:
5:19:18.912199
[19158.912198799982 Seconds]
________________________________________



## Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion (3 Layers)

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 3)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_3_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.593100,0.571631,0.781558,0.948171,0.800914,0.608426,0.409341,0.778615,0.333785,0.868344
2,0.549200,0.556925,0.774206,0.954286,0.786644,0.662949,0.449594,0.789607,0.347544,0.862394
3,0.480200,0.663805,0.845358,0.941237,0.883209,0.506815,0.390024,0.783111,0.362551,0.911300
4,0.400900,0.732138,0.835639,0.941740,0.871155,0.517968,0.389123,0.765772,0.344588,0.905074




________________________________________
Training Time:
5:20:52.385097
[19252.38509669999 Seconds]
________________________________________



# 2. Feature Extraction and LoRA (more epochs)

# Low Rank Adaption (LoRA)

# Comprehensible_FT_for_LoRA

In [37]:
Comprehensible_FT_for_LoRA = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
from peft import LoraConfig, TaskType, get_peft_model

# Instantiate in same cell model for safety. A warning appears when we apply LoRA several times to a same object.

Comprehensible_FT_for_LoRA_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    # Last 2 layers:  4, 5
    target_modules = r".*layer\.(4|5)\.attention\.(q_lin|k_lin|v_lin|out_lin)",
    bias="none",
)

Comprehensible_FT_LoRA_model = get_peft_model(Comprehensible_FT_for_LoRA, Comprehensible_FT_for_LoRA_config)
Comprehensible_FT_LoRA_model.print_trainable_parameters()



trainable params: 690,434 || all params: 136,016,644 || trainable%: 0.5076


Inspect architecture

In [39]:
print(Comprehensible_FT_LoRA_model) # See notes above on why ReLU activation does not show.

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(119547, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-3): 4 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): Linear(in_features=768, out_features=768, bias=True)
                (k_lin): Linear(in_features=768, out_features=768, bias=True)
                (v_lin): Linear(in_features=768, out_features=768, bias=True)
                (out_lin): Linear(in_features=768, out_features=768, bias=True)
              )
           

See frozen base model

In [40]:
show_trainable_and_frozen(Comprehensible_FT_LoRA_model)

Frozen: base_model.model.distilbert.embeddings.word_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.position_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: base_model.model.

Train the model

In [41]:
training_args_Comprehensible_FT_LoRA_model = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_FT_LoRA_model_3E_4_2_LAYERS",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 3e-4,
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 6,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_FT_LoRA_model_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_FT_LoRA_model,
    args              = training_args_Comprehensible_FT_LoRA_model,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_FT_LoRA_model

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
import torch

# Force out-of-memory errors instead of spilling
#torch.cuda.set_per_process_memory_fraction(0.95, device=0)

# Disable memory growth that might cause spills
#os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:False'

# Check if you're spilling to CPU
#print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


# If max_memory > 4GB, you're likely spilling or will crash

In [44]:
import torch

# Get device name
print(f"Device name: {torch.cuda.get_device_name(0)}")

# Current memory usage
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Total: {total:.2f} GB")
print(f"Free: {total - reserved:.2f} GB")
print(f"Max GPU Memory: {torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB") # peak usage

Device name: NVIDIA GeForce GTX 1650
Allocated: 0.51 GB
Reserved: 0.54 GB
Total: 4.00 GB
Free: 3.46 GB
Max GPU Memory: 0.51 GB


In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_FT_LoRA_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_FT_LoRA_model_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.495900,0.461999,0.788758,0.969878,0.790405,0.773563,0.563968,0.868761,0.453287,0.870992
2,0.437100,0.422913,0.814615,0.973559,0.816698,0.795402,0.612100,0.885374,0.500280,0.888256
3,0.429000,0.446859,0.861945,0.964720,0.879128,0.703448,0.582576,0.887112,0.527942,0.919937
4,0.401800,0.481088,0.870152,0.963563,0.889720,0.689655,0.579375,0.883926,0.528432,0.925170
5,0.381900,0.474084,0.861046,0.966726,0.876137,0.721839,0.597976,0.885793,0.524115,0.919205
6,0.373000,0.499733,0.865093,0.965235,0.882243,0.706897,0.589140,0.883278,0.522901,0.921875




________________________________________
Training Time:
4:56:54.754154
[17814.754153999966 Seconds]
________________________________________



# Repetition_FT_for_LoRA

In [39]:
Repetition_FT_for_LoRA = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
from peft import LoraConfig, TaskType, get_peft_model

# Instantiate in same cell model for safety. A warning appears when we apply LoRA several times to a same object.

Repetition_FT_for_LoRA_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    # Last 2 layers: 4, 5
    target_modules = r".*layer\.(4|5)\.attention\.(q_lin|k_lin|v_lin|out_lin)",
    bias="none",
)

Repetition_FT_LoRA_model = get_peft_model(Repetition_FT_for_LoRA, Repetition_FT_for_LoRA_config)
Repetition_FT_LoRA_model.print_trainable_parameters()



trainable params: 690,434 || all params: 136,016,644 || trainable%: 0.5076


Inspect architecture

In [41]:
print(Repetition_FT_LoRA_model) # See notes above on why ReLU activation does not show.

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(119547, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-3): 4 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): Linear(in_features=768, out_features=768, bias=True)
                (k_lin): Linear(in_features=768, out_features=768, bias=True)
                (v_lin): Linear(in_features=768, out_features=768, bias=True)
                (out_lin): Linear(in_features=768, out_features=768, bias=True)
              )
           

See frozen base model

In [42]:
show_trainable_and_frozen(Repetition_FT_LoRA_model)

Frozen: base_model.model.distilbert.embeddings.word_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.position_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: base_model.model.

Train the model

In [44]:
training_args_Repetition_FT_LoRA_model = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_FT_LoRA_model_3E_4_2_layers",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 3e-4,
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 6,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_FT_LoRA_model_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_FT_LoRA_model,
    args              = training_args_Repetition_FT_LoRA_model,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [45]:
training_args_Repetition_FT_LoRA_model

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [46]:
import torch

# Force out-of-memory errors instead of spilling
#torch.cuda.set_per_process_memory_fraction(0.95, device=0)

# Disable memory growth that might cause spills
#os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:False'

# Check if you're spilling to CPU
#print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


# If max_memory > 4GB, you're likely spilling or will crash

In [47]:
import torch

# Get device name
print(f"Device name: {torch.cuda.get_device_name(0)}")

# Current memory usage
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Total: {total:.2f} GB")
print(f"Free: {total - reserved:.2f} GB")
print(f"Max GPU Memory: {torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB") # peak usage

Device name: NVIDIA GeForce GTX 1650
Allocated: 0.51 GB
Reserved: 0.54 GB
Total: 4.00 GB
Free: 3.46 GB
Max GPU Memory: 0.51 GB


Last epoch got saved but not displayed. Best ROC at 4

In [ ]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_FT_LoRA_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_FT_LoRA_model_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.314600,0.277568,0.947414,0.978367,0.961831,0.836757,0.798588,0.960720,0.793001,0.970028
2,0.281600,0.246860,0.935452,0.984255,0.942113,0.884324,0.826437,0.962523,0.770814,0.962723
3,0.259500,0.263300,0.943053,0.981726,0.953380,0.863784,0.817164,0.962010,0.784782,0.967345
4,0.251700,0.260715,0.941184,0.981966,0.950986,0.865946,0.816932,0.965303,0.782056,0.966228
5,0.232100,0.258968,0.946168,0.982071,0.956620,0.865946,0.822566,0.964599,0.791010,0.969178


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [42]:
compute_evaluation_metrics(X_data=tokenized_repetition_validation["text"],
                           y_true_labels=tokenized_repetition_validation["labels"],
                               model_for_inference = Repetition_FT_LoRA_reloaded,
                               input_text_tokenizer=tokenizer,
                               data_loader_batch_size = 32)

{'Accuracy': 0.9465420560747664,
 'Precision': 0.9824967452625488,
 'Sensitivity (Recall, TPR)': 0.9566197183098591,
 'Specificity (TNR)': np.float64(0.8691891891891892),
 "Youden's J statistic": np.float64(0.8258089074990482),
 'ROC AUC': 0.9650257327750287,
 'Pearson correlation': np.float64(0.793074941364947),
 'F1': 0.9693855705416399}

# Grammatical_FT_for_LoRA

In [37]:
Grammatical_FT_for_LoRA = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-multilingual-cased",
    num_labels=2
)
#A warning is displayed because Head aimed for fine tuning has not yet weights.

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
from peft import LoraConfig, TaskType, get_peft_model

# Instantiate in same cell model for safety. A warning appears when we apply LoRA several times to a same object.

Grammatical_FT_for_LoRA_config = LoraConfig(
    task_type = TaskType.SEQ_CLS,
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    # Last 2 layers:  4, 5
    target_modules = r".*layer\.(4|5)\.attention\.(q_lin|k_lin|v_lin|out_lin)",
    bias="none",
)

Grammatical_FT_LoRA_model = get_peft_model(Grammatical_FT_for_LoRA, Grammatical_FT_for_LoRA_config)
Grammatical_FT_LoRA_model.print_trainable_parameters()



trainable params: 690,434 || all params: 136,016,644 || trainable%: 0.5076


Inspect architecture

In [39]:
print(Grammatical_FT_LoRA_model) # See notes above on why ReLU activation does not show.

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(119547, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-3): 4 x TransformerBlock(
              (attention): DistilBertSdpaAttention(
                (dropout): Dropout(p=0.1, inplace=False)
                (q_lin): Linear(in_features=768, out_features=768, bias=True)
                (k_lin): Linear(in_features=768, out_features=768, bias=True)
                (v_lin): Linear(in_features=768, out_features=768, bias=True)
                (out_lin): Linear(in_features=768, out_features=768, bias=True)
              )
           

See frozen base model

In [40]:
show_trainable_and_frozen(Grammatical_FT_LoRA_model)

Frozen: base_model.model.distilbert.embeddings.word_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.position_embeddings.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.weight
Frozen: base_model.model.distilbert.embeddings.LayerNorm.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: base_model.model.distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: base_model.model.distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: base_model.model.

Train the model

In [41]:
training_args_Grammatical_FT_LoRA = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_FT_LoRA_model_3E_4_",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 3e-4,
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 6,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_FT_LoRA_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_FT_LoRA_model,
    args              = training_args_Grammatical_FT_LoRA,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_FT_LoRA

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
import torch

# Force out-of-memory errors instead of spilling
#torch.cuda.set_per_process_memory_fraction(0.95, device=0)

# Disable memory growth that might cause spills
#os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:False'

# Check if you're spilling to CPU
#print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


# If max_memory > 4GB, you're likely spilling or will crash

In [44]:
import torch

# Get device name
print(f"Device name: {torch.cuda.get_device_name(0)}")

# Current memory usage
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Total: {total:.2f} GB")
print(f"Free: {total - reserved:.2f} GB")
print(f"Max GPU Memory: {torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB") # peak usage

Device name: NVIDIA GeForce GTX 1650
Allocated: 0.51 GB
Reserved: 0.54 GB
Total: 4.00 GB
Free: 3.46 GB
Max GPU Memory: 0.51 GB


In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_FT_LoRA_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_FT_LoRA_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.607600,0.606798,0.818816,0.939329,0.853699,0.506815,0.360514,0.739781,0.309729,0.894469
2,0.589000,0.578829,0.752523,0.948252,0.766694,0.625774,0.392469,0.764913,0.309472,0.847863
3,0.559700,0.587562,0.782679,0.947661,0.802715,0.603470,0.406185,0.771156,0.327131,0.869187
4,0.542700,0.571281,0.776698,0.950515,0.793017,0.630731,0.423749,0.778113,0.334573,0.864653
5,0.518100,0.610366,0.787664,0.947710,0.808534,0.600991,0.409526,0.773422,0.330234,0.872608
6,0.512800,0.619467,0.802991,0.946884,0.827376,0.584882,0.412258,0.774063,0.334932,0.883105




________________________________________
Training Time:
2:03:49.999614
[7429.999613800028 Seconds]
________________________________________



# Feature Extraction 12 epochs

## Comprehensible

In [37]:
Comprehensible_DistilBertLast3LayerFusion_config = DistilBertConfig.from_pretrained(
                                                    "distilbert-base-multilingual-cased",
                                                    num_labels=2,
                                                    dropout=0.1,
                                                    attention_dropout=0.1,
                                                    seq_classif_dropout=0.1,
                                                )

Comprehensible_DistilBertLast3LayerFusion_model = DistilBertLast3LayerFusion.from_pretrained(
    "distilbert-base-multilingual-cased",
    config=Comprehensible_DistilBertLast3LayerFusion_config,
    ignore_mismatched_sizes=True  # important for custom head
)


Some weights of DistilBertLast3LayerFusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBertLast3LayerFusion_model) # See notes above on why ReLU activation does not show.

DistilBertLast3LayerFusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

Freeze all DestilBERT. We use only as Feature Extractor.

In [39]:
freeze_all_destilbert(Comprehensible_DistilBertLast3LayerFusion_model)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBertLast3LayerFusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Comprehensible_DistilBertLast3LayerFusion_model = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_FE_12_epochs",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 12,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBertLast3LayerFusion_model_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBertLast3LayerFusion_model,
    args              = training_args_Comprehensible_DistilBertLast3LayerFusion_model,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_DistilBertLast3LayerFusion_model

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBertLast3LayerFusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBertLast3LayerFusion_model_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.505800,0.501236,0.748510,0.971643,0.742928,0.800000,0.542928,0.848536,0.423143,0.842031
2,0.454300,0.452621,0.836537,0.965434,0.849221,0.719540,0.568761,0.869546,0.487151,0.903606
3,0.446700,0.482436,0.878471,0.959746,0.903178,0.650575,0.553752,0.875728,0.524403,0.930603
4,0.419700,0.439032,0.828668,0.968439,0.837383,0.748276,0.585659,0.878286,0.497201,0.898156
5,0.404200,0.430500,0.853513,0.967325,0.866916,0.729885,0.596801,0.883991,0.519008,0.914372
6,0.406200,0.437199,0.852501,0.969113,0.864050,0.745977,0.610027,0.882263,0.512450,0.913570
7,0.394400,0.450977,0.871838,0.964136,0.891090,0.694253,0.585343,0.884040,0.535648,0.926175
8,0.393100,0.434424,0.849129,0.968194,0.861059,0.739080,0.600140,0.884516,0.512465,0.911489
9,0.380700,0.431313,0.835413,0.971133,0.842617,0.768966,0.611582,0.887523,0.509858,0.902322
10,0.382800,0.448625,0.860483,0.965550,0.876636,0.711494,0.588130,0.885660,0.525287,0.918947




________________________________________
Training Time:
4:26:42.133486
[16002.133485800005 Seconds]
________________________________________



## Repetition

In [37]:
Repetition_free_DistilBertLast3LayerFusion_config = DistilBertConfig.from_pretrained(
                                                    "distilbert-base-multilingual-cased",
                                                    num_labels=2,
                                                    dropout=0.1,
                                                    attention_dropout=0.1,
                                                    seq_classif_dropout=0.1,
                                                )

Repetition_free_DistilBertLast3LayerFusion_model = DistilBertLast3LayerFusion.from_pretrained(
    "distilbert-base-multilingual-cased",
    config=Repetition_free_DistilBertLast3LayerFusion_config,
    ignore_mismatched_sizes=True  # important for custom head
)


Some weights of DistilBertLast3LayerFusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_free_DistilBertLast3LayerFusion_model) # See notes above on why ReLU activation does not show.

DistilBertLast3LayerFusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

Freeze all DestilBERT. We use only as Feature Extractor.

In [39]:
freeze_all_destilbert(Repetition_free_DistilBertLast3LayerFusion_model)

In [40]:
show_trainable_and_frozen(Repetition_free_DistilBertLast3LayerFusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_free_DistilBertLast3LayerFusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_free_FE_12_epochs",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 12,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_free_DistilBertLast3LayerFusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_free_DistilBertLast3LayerFusion_model,
    args              = training_args_Repetition_free_DistilBertLast3LayerFusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_free_DistilBertLast3LayerFusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_free_DistilBertLast3LayerFusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_free_DistilBertLast3LayerFusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.354100,0.313346,0.912897,0.974078,0.926197,0.810811,0.737008,0.940094,0.693187,0.949534
2,0.323800,0.294633,0.902679,0.979948,0.908592,0.857297,0.765889,0.948736,0.689377,0.942922
3,0.313900,0.286331,0.894206,0.982256,0.896620,0.875676,0.772295,0.953460,0.685004,0.937486
4,0.299800,0.288116,0.898816,0.982209,0.901972,0.874595,0.776566,0.955073,0.691710,0.940382
5,0.288800,0.292238,0.883614,0.982473,0.884225,0.878919,0.763144,0.954682,0.670317,0.930764
6,0.269200,0.283296,0.890467,0.983823,0.890845,0.887568,0.778413,0.956621,0.680864,0.935028
7,0.263300,0.290736,0.895327,0.982429,0.897746,0.876757,0.774503,0.954563,0.683456,0.938181
8,0.263400,0.278564,0.903551,0.982753,0.906901,0.877838,0.784739,0.957803,0.703589,0.943305
9,0.244400,0.280301,0.911277,0.981314,0.917183,0.865946,0.783129,0.957203,0.711909,0.948165
10,0.257200,0.278104,0.918255,0.981039,0.925493,0.862703,0.788196,0.958042,0.726510,0.952457




________________________________________
Training Time:
3:06:56.551479
[11216.5514789 Seconds]
________________________________________



## Grammatical

In [37]:
Grammatical_DistilBertLast3LayerFusion_config = DistilBertConfig.from_pretrained(
                                                    "distilbert-base-multilingual-cased",
                                                    num_labels=2,
                                                    dropout=0.1,
                                                    attention_dropout=0.1,
                                                    seq_classif_dropout=0.1,
                                                )

Grammatical_DistilBertLast3LayerFusion_model = DistilBertLast3LayerFusion.from_pretrained(
    "distilbert-base-multilingual-cased",
    config=Grammatical_DistilBertLast3LayerFusion_config,
    ignore_mismatched_sizes=True  # important for custom head
)


Some weights of DistilBertLast3LayerFusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBertLast3LayerFusion_model) # See notes above on why ReLU activation does not show.

DistilBertLast3LayerFusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

Freeze all DestilBERT. We use only as Feature Extractor.

In [39]:
freeze_all_destilbert(Grammatical_DistilBertLast3LayerFusion_model)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBertLast3LayerFusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Grammatical_DistilBertLast3LayerFusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_FE_12_epochs",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 12,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBertLast3LayerFusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBertLast3LayerFusion_model,
    args              = training_args_Grammatical_DistilBertLast3LayerFusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Grammatical_DistilBertLast3LayerFusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBertLast3LayerFusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBertLast3LayerFusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.618000,0.608488,0.783801,0.940543,0.810889,0.541512,0.352401,0.732007,0.286811,0.870917
2,0.608000,0.595022,0.737321,0.946366,0.750485,0.619579,0.370064,0.744323,0.286028,0.837119
3,0.585600,0.590060,0.766480,0.946972,0.784289,0.607187,0.391476,0.753242,0.307411,0.857987
4,0.578500,0.584987,0.777321,0.944508,0.799390,0.579926,0.379316,0.757053,0.307178,0.865911
5,0.558300,0.615570,0.781184,0.944499,0.803962,0.577447,0.381410,0.748333,0.302630,0.868583
6,0.563000,0.593145,0.787414,0.943515,0.812275,0.565056,0.377331,0.758350,0.311554,0.872990
7,0.544100,0.583625,0.739439,0.950923,0.748961,0.654275,0.403236,0.761766,0.306137,0.837945
8,0.536500,0.603710,0.815576,0.942201,0.846911,0.535316,0.382226,0.761234,0.324078,0.892018
9,0.534700,0.600940,0.785545,0.944525,0.809088,0.574969,0.384057,0.760219,0.314582,0.871577
10,0.520600,0.611854,0.816822,0.941475,0.849127,0.527881,0.377008,0.761835,0.323024,0.892920




________________________________________
Training Time:
2:54:11.541531
[10451.541531099996 Seconds]
________________________________________



# Learning Rate (Comprehensible)

In [45]:
#5e-5 linear 0.3322

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.474700,0.449394,0.819674,0.969990,0.825670,0.764368,0.590038,0.874103,0.484506,0.892030
2,0.409100,0.410803,0.830467,0.972178,0.836012,0.779310,0.615323,0.893640,0.511287,0.898968
3,0.370800,0.475154,0.869477,0.965167,0.887352,0.704598,0.591950,0.889116,0.535220,0.924625
4,0.332200,0.479402,0.871276,0.965242,0.889346,0.704598,0.593943,0.892094,0.535608,0.925741




________________________________________
Training Time:
2:01:38.080620
[7298.080620199908 Seconds]
________________________________________



## 3e-5 cosine Loss 0.345000

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_Concat_LR_3E_5",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 3e-5, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.470200,0.447615,0.829342,0.968601,0.838006,0.749425,0.587432,0.873248,0.490013,0.898584
2,0.409900,0.417778,0.828893,0.972533,0.833894,0.782759,0.616653,0.891059,0.507299,0.897893
3,0.375700,0.465277,0.865880,0.966667,0.881745,0.719540,0.601285,0.889322,0.532089,0.922255
4,0.345000,0.450598,0.854750,0.969199,0.866542,0.745977,0.612519,0.892385,0.523668,0.915000




________________________________________
Training Time:
2:23:02.708494
[8582.708494099992 Seconds]
________________________________________



## 1e-5 linear 0.399000

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_Concat_LR_1E_5",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-5, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "linear", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.482500,0.461511,0.831591,0.965217,0.843738,0.719540,0.563279,0.863190,0.477146,0.900399
2,0.428000,0.441272,0.828893,0.969666,0.836511,0.758621,0.595132,0.877727,0.493174,0.898180
3,0.422300,0.455318,0.849916,0.966918,0.863178,0.727586,0.590764,0.878072,0.512184,0.912107
4,0.399000,0.450443,0.846543,0.967697,0.858567,0.735632,0.594199,0.881245,0.511457,0.909871




________________________________________
Training Time:
2:22:10.731256
[8530.731256400002 Seconds]
________________________________________



## 1e-4 cosine 0.319100

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [44]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_Concat_LR_1E_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [45]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [46]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.476000,0.458251,0.810455,0.970741,0.814455,0.773563,0.588018,0.876210,0.477634,0.885757
2,0.413000,0.413037,0.833614,0.972563,0.839252,0.781609,0.620862,0.892478,0.516519,0.901003
3,0.372100,0.495359,0.877684,0.963270,0.898692,0.683908,0.582600,0.890388,0.541147,0.929861
4,0.319100,0.502448,0.871276,0.963113,0.891464,0.685057,0.576522,0.890909,0.532229,0.925904




________________________________________
Training Time:
2:46:35.299739
[9995.299739300011 Seconds]
________________________________________



## 3e-4 cosine 0.373600

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [43]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_Concat_LR_3E_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 3e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [44]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.501400,0.466781,0.761327,0.976429,0.753645,0.832184,0.585829,0.868315,0.442722,0.850693
2,0.448600,0.454254,0.845194,0.967905,0.856822,0.737931,0.594753,0.873966,0.513021,0.908983
3,0.426500,0.521217,0.870601,0.963708,0.890093,0.690805,0.580898,0.881863,0.521243,0.925439
4,0.373600,0.512036,0.862619,0.965385,0.879252,0.709195,0.588448,0.880094,0.517053,0.920308




________________________________________
Training Time:
2:14:56.577983
[8096.577983399999 Seconds]
________________________________________



## 1.5e-4 cosine 0.326900

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Comprehensible_Concat_LR_1_5E_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1.5e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.1, # SAME FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.483300,0.467068,0.801237,0.971657,0.803115,0.783908,0.587023,0.872824,0.472991,0.879383
2,0.426300,0.429151,0.832265,0.972516,0.837757,0.781609,0.619366,0.885998,0.506058,0.900120
3,0.386800,0.505615,0.878583,0.963433,0.899564,0.685057,0.584621,0.885201,0.538516,0.930403
4,0.326900,0.501624,0.873300,0.965201,0.891713,0.703448,0.595162,0.888367,0.531883,0.927003




________________________________________
Training Time:
2:15:10.546371
[8110.546371100005 Seconds]
________________________________________



# Learning Rate (Sum Fusion Repetition)

In [44]:
# 5 x 10^-5

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.298900,0.258893,0.945047,0.981490,0.955915,0.861622,0.817537,0.965142,0.790786,0.968534
2,0.247600,0.247562,0.943427,0.982013,0.953521,0.865946,0.819467,0.965721,0.788698,0.967558
3,0.216000,0.274836,0.942181,0.982548,0.951549,0.870270,0.821820,0.964173,0.782113,0.966800
4,0.193900,0.294793,0.947165,0.981813,0.958028,0.863784,0.821812,0.962250,0.789183,0.969775




________________________________________
Training Time:
1:21:23.758586
[4883.7585862998385 Seconds]
________________________________________



### 1e-4 linear 0.156500

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_LR_search_1",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.290200,0.266017,0.947539,0.981404,0.958873,0.860541,0.819414,0.965359,0.797491,0.970008
2,0.238500,0.246234,0.942555,0.983399,0.951127,0.876757,0.827884,0.965790,0.788465,0.966994
3,0.201100,0.296659,0.945545,0.980944,0.957042,0.857297,0.814340,0.962405,0.788030,0.968846
4,0.156500,0.351475,0.947913,0.978653,0.962113,0.838919,0.801032,0.958179,0.788993,0.970313




________________________________________
Training Time:
1:19:09.077687
[4749.077686700039 Seconds]
________________________________________



### 1e-5 linear 0.253500

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [43]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_LR_search_1_e_5_linear",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-5,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [44]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.322400,0.280569,0.921745,0.980261,0.930282,0.856216,0.786498,0.952500,0.723843,0.954618
2,0.275000,0.263234,0.926106,0.982071,0.933521,0.869189,0.802710,0.960833,0.747500,0.957181
3,0.263400,0.265051,0.933832,0.982518,0.941972,0.871351,0.813323,0.961633,0.762370,0.961818
4,0.253500,0.263448,0.932835,0.982924,0.940423,0.874595,0.815017,0.962942,0.760245,0.961203




________________________________________
Training Time:
1:21:53.362773
[4913.36277300003 Seconds]
________________________________________



### 3e-4 linear 0.196000

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_LR_search_Sum_3e_4_linear",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 3e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.305200,0.315258,0.937072,0.980615,0.947606,0.856216,0.803822,0.960016,0.763067,0.963828
2,0.264400,0.259727,0.944548,0.980922,0.955915,0.857297,0.813213,0.963109,0.786921,0.968257
3,0.230500,0.280205,0.942430,0.980875,0.953521,0.857297,0.810818,0.961447,0.774910,0.967005
4,0.196000,0.313245,0.951402,0.978875,0.965915,0.840000,0.805915,0.960601,0.796819,0.972352




________________________________________
Training Time:
1:32:22.743587
[5542.743587199948 Seconds]
________________________________________



### 7.5e-5 cosine 	0.164200

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_LR_search_7_5_e_5",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 7.5e-5,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "cosine",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.290600,0.260577,0.948037,0.982109,0.958732,0.865946,0.824678,0.965780,0.797210,0.970280
2,0.240500,0.248329,0.941807,0.983525,0.950141,0.877838,0.827979,0.965662,0.784521,0.966545
3,0.197800,0.293794,0.944798,0.981484,0.955634,0.861622,0.817255,0.963260,0.782220,0.968386
4,0.164200,0.332924,0.948411,0.977708,0.963662,0.831351,0.795013,0.960347,0.791702,0.970634




________________________________________
Training Time:
1:20:01.920017
[4801.920017099939 Seconds]
________________________________________



# Learning Rate (Grammatical)

In [44]:
# 1e-5 linear

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.602200,0.589330,0.816324,0.940897,0.849127,0.522924,0.372052,0.759668,0.324854,0.892659
2,0.563500,0.559085,0.757508,0.955110,0.766417,0.677819,0.444236,0.785188,0.335731,0.850423
3,0.516500,0.601657,0.825171,0.944232,0.856193,0.547708,0.403900,0.786017,0.357828,0.898060
4,0.475700,0.606159,0.814704,0.945161,0.842893,0.562577,0.405470,0.782092,0.349396,0.891102




________________________________________
Training Time:
1:21:42.271721
[4902.271721499972 Seconds]
________________________________________



## 1e-4 linear 0.439800

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1E_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.602300,0.584608,0.821558,0.940737,0.855500,0.517968,0.373468,0.765675,0.315743,0.896096
2,0.564000,0.559408,0.758380,0.953756,0.768634,0.666667,0.435301,0.783955,0.337804,0.851247
3,0.506500,0.619090,0.836885,0.942622,0.871710,0.525403,0.397112,0.776579,0.358203,0.905780
4,0.439800,0.690948,0.827414,0.943304,0.859795,0.537794,0.397589,0.763121,0.341717,0.899616




________________________________________
Training Time:
1:24:31.149883
[5071.149882599944 Seconds]
________________________________________



## 7.5E-5 0.472900

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [40]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [44]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_7_5_e_5",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 7.5e-5,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [45]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [46]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.601000,0.593201,0.822430,0.939196,0.858132,0.503098,0.361230,0.757644,0.326441,0.896836
2,0.568700,0.572728,0.772960,0.949667,0.789415,0.625774,0.415190,0.772595,0.324445,0.862158
3,0.517500,0.609346,0.817196,0.944642,0.846356,0.556382,0.402738,0.779235,0.346900,0.892802
4,0.472900,0.633520,0.816698,0.945297,0.845109,0.562577,0.407687,0.776248,0.343356,0.892400




________________________________________
Training Time:
1:32:58.231635
[5578.231634699972 Seconds]
________________________________________



## 1.5E-4 cosine 0.448200

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1_5_E_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1.5e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "cosine",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.605400,0.587273,0.790530,0.944453,0.815046,0.571252,0.386297,0.765836,0.300327,0.874991
2,0.564200,0.560358,0.759502,0.953050,0.770574,0.660471,0.431044,0.784285,0.335996,0.852153
3,0.502800,0.634940,0.824050,0.944020,0.855085,0.546468,0.401553,0.779076,0.353536,0.897354
4,0.448200,0.671876,0.828037,0.944428,0.859379,0.547708,0.407087,0.772653,0.349608,0.899898




________________________________________
Training Time:
1:40:36.363063
[6036.363062900025 Seconds]
________________________________________



## 3E-4 cosine 0.554700

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_3_E_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 3e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "cosine",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.619600,0.642423,0.880872,0.920494,0.949571,0.266419,0.215989,0.696399,0.259056,0.934806
2,0.608800,0.597029,0.820561,0.940128,0.854946,0.513011,0.367957,0.742973,0.316633,0.895516
3,0.569700,0.613657,0.830530,0.939790,0.867138,0.503098,0.370236,0.760353,0.323654,0.902003
4,0.554700,0.614722,0.827290,0.940350,0.862704,0.510533,0.373237,0.761143,0.325977,0.899855




________________________________________
Training Time:
2:12:05.563150
[7925.563149700058 Seconds]
________________________________________



# Regularization

## Comprehensible Concat 2 Layers

0.892478

In [46]:
# 1e-4

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.476000,0.458251,0.810455,0.970741,0.814455,0.773563,0.588018,0.876210,0.477634,0.885757
2,0.413000,0.413037,0.833614,0.972563,0.839252,0.781609,0.620862,0.892478,0.516519,0.901003
3,0.372100,0.495359,0.877684,0.963270,0.898692,0.683908,0.582600,0.890388,0.541147,0.929861
4,0.319100,0.502448,0.871276,0.963113,0.891464,0.685057,0.576522,0.890909,0.532229,0.925904




________________________________________
Training Time:
2:46:35.299739
[9995.299739300011 Seconds]
________________________________________



## 0.892266

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.1,  # Embeddings
    attention_dropout   = 0.2,  # Transformer
    seq_classif_dropout = 0.4,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.2, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Reg_search_comprehensible_1",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.05  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.480100,0.462368,0.814727,0.970211,0.819813,0.767816,0.587629,0.874839,0.479275,0.888694
2,0.427200,0.413941,0.835301,0.972622,0.841121,0.781609,0.622731,0.890832,0.513134,0.902105
3,0.384800,0.481870,0.875323,0.966289,0.892960,0.712644,0.605603,0.890892,0.538491,0.928178
4,0.340900,0.483709,0.868016,0.965991,0.884860,0.712644,0.597503,0.892266,0.532610,0.923647




________________________________________
Training Time:
2:09:34.098082
[7774.0980824 Seconds]
________________________________________



## 0.890647

In [42]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.2,  # Embeddings
    attention_dropout   = 0.25,  # Transformer
    seq_classif_dropout = 0.4,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [43]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.25, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inpl

Freeze 4 layers

In [44]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [45]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [46]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Reg_search_comprehensible_2",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.1  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [47]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [48]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.484000,0.469395,0.804384,0.970645,0.807601,0.774713,0.582314,0.873290,0.467991,0.881649
2,0.430900,0.419473,0.819337,0.974845,0.820935,0.804598,0.625532,0.890647,0.501665,0.891294
3,0.406600,0.475526,0.861495,0.966616,0.876760,0.720690,0.597450,0.888967,0.534389,0.919498
4,0.368500,0.476912,0.850253,0.969816,0.860810,0.752874,0.613684,0.889850,0.520827,0.912068




________________________________________
Training Time:
1:57:49.673946
[7069.6739462000005 Seconds]
________________________________________



## 0.890974

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.15,  # Embeddings
    attention_dropout   = 0.1,  # Transformer
    seq_classif_dropout = 0.3,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.15, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.15, inp

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Reg_search_comprehensible_3",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.005  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.475600,0.461937,0.801911,0.972394,0.803240,0.789655,0.592895,0.876222,0.471701,0.879760
2,0.419900,0.415792,0.812816,0.975052,0.813333,0.808046,0.621379,0.890332,0.501106,0.886881
3,0.383900,0.492476,0.870377,0.965709,0.887850,0.709195,0.597046,0.889618,0.535549,0.925144
4,0.334700,0.492255,0.862170,0.966004,0.878131,0.714943,0.593073,0.890974,0.525359,0.919974




________________________________________
Training Time:
2:06:08.237271
[7568.237270900001 Seconds]
________________________________________



## 0.891234

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.05,  # Embeddings
    attention_dropout   = 0.05,  # Transformer
    seq_classif_dropout = 0.1,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.05, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.05, in

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [43]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Reg_search_comprehensible_low_reg",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.005  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [44]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.468600,0.446018,0.808094,0.973755,0.809097,0.798851,0.607947,0.879307,0.481120,0.883822
2,0.403000,0.416307,0.829455,0.972553,0.834517,0.782759,0.617276,0.891234,0.511756,0.898263
3,0.351100,0.519725,0.883530,0.960227,0.908536,0.652874,0.561409,0.884314,0.538530,0.933666
4,0.280000,0.574016,0.883080,0.959842,0.908411,0.649425,0.557837,0.883472,0.532020,0.933419




________________________________________
Training Time:
2:26:40.523147
[8800.523146900003 Seconds]
________________________________________



## 0.893530

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.075,  # Embeddings
    attention_dropout   = 0.075,  # Transformer
    seq_classif_dropout = 0.25,  # Classification head
)

# Load model with custom config
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Concat_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Concat_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Concat_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.075, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.075, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.075,

Freeze 4 layers

In [39]:
freeze_first_n_layers(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [43]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Reg_search_comprehensible_0025_wd_01",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4, # LOWERED FROM 5e-5
    warmup_ratio  = 0.05, # LOWERED FROM 0.1
    lr_scheduler_type= "cosine", #changed from Linear

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.05  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model,
    args              = training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion,
    train_dataset     = tokenized_comprehensible_train,
    eval_dataset      = tokenized_comprehensible_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_comprehensible,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [44]:
training_args_Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [45]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Comprehensible_DistilBert_CLS_and_AVG_Pooling_Concat_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.475600,0.449109,0.829117,0.970899,0.835639,0.768966,0.604604,0.880794,0.492116,0.898205
2,0.415000,0.410779,0.828780,0.975015,0.831526,0.803448,0.634975,0.893530,0.511533,0.897572
3,0.376100,0.497709,0.878471,0.962687,0.900187,0.678161,0.578348,0.890083,0.539673,0.930388
4,0.313300,0.510080,0.873637,0.962720,0.894579,0.680460,0.575039,0.890070,0.532944,0.927400




________________________________________
Training Time:
2:07:31.384415
[7651.384415200009 Seconds]
________________________________________



# Repetition

In [43]:
# previous

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.290200,0.266017,0.947539,0.981404,0.958873,0.860541,0.819414,0.965359,0.797491,0.970008
2,0.238500,0.246234,0.942555,0.983399,0.951127,0.876757,0.827884,0.965790,0.788465,0.966994
3,0.201100,0.296659,0.945545,0.980944,0.957042,0.857297,0.814340,0.962405,0.788030,0.968846
4,0.156500,0.351475,0.947913,0.978653,0.962113,0.838919,0.801032,0.958179,0.788993,0.970313




________________________________________
Training Time:
1:19:09.077687
[4749.077686700039 Seconds]
________________________________________



### 0.965335

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.2,  # Embeddings
    attention_dropout   = 0.2,  # Transformer
    seq_classif_dropout = 0.4,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.2, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_REG_search_1_02",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.005  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.320600,0.259148,0.918255,0.985387,0.921268,0.895135,0.816403,0.963839,0.730769,0.952249
2,0.271500,0.248956,0.916885,0.986243,0.918873,0.901622,0.820495,0.964325,0.734034,0.951367
3,0.235600,0.268907,0.918505,0.986124,0.920845,0.900541,0.821386,0.965335,0.730790,0.952367
4,0.213300,0.282910,0.929346,0.983711,0.935634,0.881081,0.816715,0.963093,0.748731,0.959070




________________________________________
Training Time:
1:53:05.480614
[6785.480614300002 Seconds]
________________________________________



### 0.964389

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.15,  # Embeddings
    attention_dropout   = 0.15,  # Transformer
    seq_classif_dropout = 0.4,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [43]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.15, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.15, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.15, inpla

Freeze 4 layers

In [44]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [45]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [46]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_REG_search_2",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [47]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [48]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.307400,0.251641,0.933707,0.983083,0.941268,0.875676,0.816943,0.964389,0.766775,0.961721
2,0.254600,0.255871,0.930093,0.984012,0.936197,0.883243,0.819440,0.963827,0.756527,0.959509
3,0.222200,0.297501,0.941682,0.981837,0.951690,0.864865,0.816555,0.963033,0.772290,0.966528
4,0.190900,0.324768,0.944922,0.980237,0.957042,0.851892,0.808934,0.959285,0.778967,0.968501




________________________________________
Training Time:
1:49:37.667251
[6577.667250899998 Seconds]
________________________________________



### 0.965403

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.05,  # Embeddings
    attention_dropout   = 0.05,  # Transformer
    seq_classif_dropout = 0.2,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.05, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.05, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_REG_search_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.05  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.285300,0.285358,0.951651,0.979155,0.965915,0.842162,0.808078,0.965058,0.807751,0.972490
2,0.227400,0.242805,0.947539,0.982936,0.957324,0.872432,0.829756,0.965403,0.799363,0.969961
3,0.177300,0.315157,0.949657,0.980758,0.961972,0.855135,0.817107,0.960977,0.796288,0.971274
4,0.122300,0.399967,0.955140,0.975989,0.973239,0.816216,0.789456,0.954733,0.804710,0.974612




________________________________________
Training Time:
1:40:31.894969
[6031.894969300003 Seconds]
________________________________________



### 0.962755

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.25,  # Embeddings
    attention_dropout   = 0.25,  # Transformer
    seq_classif_dropout = 0.25,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.25, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.25, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.25, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_REG_search_5",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.1  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.341300,0.288310,0.872523,0.986861,0.867465,0.911351,0.778816,0.957751,0.658937,0.923319
2,0.297200,0.258143,0.901308,0.985530,0.901690,0.898378,0.800069,0.961228,0.704679,0.941748
3,0.265400,0.278896,0.900810,0.986871,0.899859,0.908108,0.807967,0.961555,0.697429,0.941358
4,0.245800,0.280104,0.907165,0.986675,0.907324,0.905946,0.813270,0.962755,0.707124,0.945337




________________________________________
Training Time:
1:37:20.720505
[5840.720504500001 Seconds]
________________________________________



### 0.962755

In [43]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.3,  # Embeddings
    attention_dropout   = 0.3,  # Transformer
    seq_classif_dropout = 0.3,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [44]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.3, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.3, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.3, inplace=

Freeze 4 layers

In [45]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [46]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [47]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_REG_search_6",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.05  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [48]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [49]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.376000,0.344276,0.801994,0.989345,0.784648,0.935135,0.719783,0.954773,0.592697,0.875187
2,0.322000,0.339706,0.805607,0.989745,0.788451,0.937297,0.725748,0.957882,0.593029,0.877705
3,0.302000,0.311124,0.856698,0.988827,0.847606,0.926486,0.774092,0.959683,0.638981,0.912786
4,0.283600,0.317385,0.852212,0.989084,0.842254,0.928649,0.770902,0.960274,0.631763,0.909782




________________________________________
Training Time:
1:32:51.826208
[5571.826208099999 Seconds]
________________________________________



### 0.966313

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.075,  # Embeddings
    attention_dropout   = 0.075,  # Transformer
    seq_classif_dropout = 0.25,  # Classification head
)

# Load model with custom config
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.075, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.075, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.075, in

Freeze 4 layers

In [39]:
freeze_first_n_layers(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Repetition_REG_search_wd_01_d_0025",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.1  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_repetition_train,
    eval_dataset      = tokenized_repetition_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_repetition,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Repetition_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.290000,0.275266,0.948037,0.980999,0.959859,0.857297,0.817156,0.965179,0.796969,0.970314
2,0.237300,0.248423,0.945171,0.982469,0.955070,0.869189,0.824260,0.966313,0.795471,0.968576
3,0.186200,0.296727,0.943676,0.982298,0.953521,0.868108,0.821629,0.963196,0.781865,0.967696
4,0.140600,0.362264,0.953520,0.977974,0.969296,0.832432,0.801728,0.959124,0.801961,0.973615




________________________________________
Training Time:
1:21:15.606358
[4875.606358299992 Seconds]
________________________________________



# Grammatical

In [43]:
#previous

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.602300,0.584608,0.821558,0.940737,0.855500,0.517968,0.373468,0.765675,0.315743,0.896096
2,0.564000,0.559408,0.758380,0.953756,0.768634,0.666667,0.435301,0.783955,0.337804,0.851247
3,0.506500,0.619090,0.836885,0.942622,0.871710,0.525403,0.397112,0.776579,0.358203,0.905780
4,0.439800,0.690948,0.827414,0.943304,0.859795,0.537794,0.397589,0.763121,0.341717,0.899616




________________________________________
Training Time:
1:24:31.149883
[5071.149882599944 Seconds]
________________________________________



## 0.781895

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.2,  # Embeddings
    attention_dropout   = 0.2,  # Transformer
    seq_classif_dropout = 0.3,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.2, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1E_4_REG_1",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.05  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.605400,0.598683,0.845732,0.934467,0.890967,0.441140,0.332107,0.749714,0.321390,0.912199
2,0.581800,0.570909,0.790031,0.947292,0.811721,0.596035,0.407755,0.779706,0.337204,0.874282
3,0.537400,0.629540,0.836012,0.941238,0.872125,0.513011,0.385136,0.780976,0.353675,0.905365
4,0.511500,0.626379,0.821059,0.946280,0.849266,0.568773,0.418039,0.781895,0.352626,0.895152




________________________________________
Training Time:
1:35:19.161821
[5719.161820699999 Seconds]
________________________________________



## 0.786065

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.05,  # Embeddings
    attention_dropout   = 0.05,  # Transformer
    seq_classif_dropout = 0.05,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.05, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.05, inpla

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1E_4_REG_2",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.592100,0.580795,0.799128,0.944638,0.825021,0.567534,0.392555,0.769491,0.320360,0.880787
2,0.552900,0.559376,0.756636,0.953176,0.767110,0.662949,0.430059,0.786065,0.336644,0.850081
3,0.483400,0.655953,0.821308,0.941932,0.853976,0.529120,0.383096,0.776837,0.341737,0.895800
4,0.389900,0.780821,0.828411,0.940564,0.863813,0.511772,0.375585,0.758770,0.329165,0.900556




________________________________________
Training Time:
1:45:36.339059
[6336.339058500002 Seconds]
________________________________________



## 1e-4 linear 0.786600

In [38]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.15,  # Embeddings
    attention_dropout   = 0.15,  # Transformer
    seq_classif_dropout = 0.15,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [39]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.15, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.15, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.15, inpla

Freeze 4 layers

In [40]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [41]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1E_4_REG_3",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.03  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [43]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [44]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.605800,0.585038,0.830156,0.939235,0.867276,0.498141,0.365418,0.762903,0.326313,0.901822
2,0.567800,0.559202,0.756386,0.953472,0.766556,0.665428,0.431983,0.785225,0.337984,0.849858
3,0.521300,0.610433,0.829283,0.941825,0.863536,0.522924,0.386460,0.786600,0.361328,0.900983
4,0.478400,0.626134,0.816449,0.946248,0.843863,0.571252,0.415114,0.783550,0.353995,0.892127




________________________________________
Training Time:
1:31:05.777285
[5465.777284699987 Seconds]
________________________________________



## 1e-4 linear 0.787007

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.075,  # Embeddings
    attention_dropout   = 0.075,  # Transformer
    seq_classif_dropout = 0.25,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.075, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.075, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.075, in

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1E_4_REG_4",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.1  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.596600,0.581968,0.836012,0.939922,0.873511,0.500620,0.374130,0.771006,0.327156,0.905501
2,0.557400,0.556742,0.756262,0.952996,0.766833,0.661710,0.428543,0.787007,0.341517,0.849839
3,0.496700,0.657783,0.839875,0.942299,0.875589,0.520446,0.396035,0.784382,0.359690,0.907720
4,0.415800,0.682124,0.830903,0.943679,0.863536,0.539033,0.402569,0.776794,0.353557,0.901830




________________________________________
Training Time:
1:22:40.618588
[4960.618587600009 Seconds]
________________________________________



## 0.785506

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.015,  # Embeddings
    attention_dropout   = 0.015,  # Transformer
    seq_classif_dropout = 0.25,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.015, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.015, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.015, in

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1E_4_REG_5",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.15  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.592300,0.578001,0.836885,0.939462,0.875035,0.495663,0.370698,0.777684,0.334980,0.906104
2,0.550400,0.559535,0.752523,0.953537,0.761984,0.667906,0.429890,0.785506,0.336712,0.847066
3,0.473700,0.664655,0.840623,0.942087,0.876697,0.517968,0.394665,0.782647,0.359611,0.908217
4,0.346800,0.853460,0.842617,0.940915,0.880299,0.505576,0.385875,0.749326,0.336202,0.909598




________________________________________
Training Time:
1:26:09.429851
[5169.429850799992 Seconds]
________________________________________



## 0.786270

In [37]:
from transformers import DistilBertForSequenceClassification, DistilBertConfig

# Create custom config with dropout values
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config = DistilBertConfig.from_pretrained(
    'distilbert-base-multilingual-cased',
    num_labels          = 2,    # Binary classification
    dropout             = 0.025,  # Embeddings
    attention_dropout   = 0.025,  # Transformer
    seq_classif_dropout = 0.025,  # Classification head
)

# Load model with custom config
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model = DistilBert_CLS_and_AVG_Pooling_Sum_Fusion.from_pretrained(
                                                   'distilbert-base-multilingual-cased',
                                                    config=Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_config,
                                                    ignore_mismatched_sizes=True  # Important! Since we have custom layers
)

Some weights of DistilBert_CLS_and_AVG_Pooling_Sum_Fusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['avg_head.0.bias', 'avg_head.0.weight', 'cls_head.0.bias', 'cls_head.0.weight', 'merge_classifier.bias', 'merge_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [38]:
print(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model) # See notes above on why ReLU activation does not show.

DistilBert_CLS_and_AVG_Pooling_Sum_Fusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.025, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.025, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.025, in

Freeze 4 layers

In [39]:
freeze_first_n_layers(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model, 4)

In [40]:
show_trainable_and_frozen(Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [41]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion = TrainingArguments(

    output_dir                 = "Saved_models/Grammatical_Sum_LR_1E_4_REG_6",
    eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    learning_rate = 1e-4,
    warmup_ratio  = 0.05,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.15  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model,
    args              = training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion,
    train_dataset     = tokenized_grammar_train,
    eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = inverse_frequency_class_weights_grammar,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [42]:
training_args_Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [43]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBert_CLS_and_AVG_Pooling_Sum_Fusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,"Sensitivity (recall, tpr)",Specificity (tnr),Youden's j statistic,Roc auc,Pearson correlation,F1
1,0.594200,0.580471,0.800374,0.946991,0.824190,0.587361,0.411550,0.773079,0.323087,0.881333
2,0.548200,0.558731,0.766106,0.952550,0.778748,0.653036,0.431784,0.786270,0.339951,0.856925
3,0.472500,0.660217,0.838380,0.940616,0.875589,0.505576,0.381165,0.777058,0.350246,0.906938
4,0.354700,0.861608,0.841121,0.936921,0.882793,0.468401,0.351195,0.734455,0.315848,0.909052




________________________________________
Training Time:
1:22:50.792892
[4970.792891999998 Seconds]
________________________________________



# MELA

# MELA

In [1]:

import zipfile
import os
import wget
from datasets import load_dataset,Dataset
import os
from pathlib import Path
import pandas as pd

url =  "https://github.com/sjtu-compling/MELA/raw/refs/heads/main/data.zip"
filename = wget.download(url)

os.makedirs("MELA", exist_ok=True)
password = "200240"
with zipfile.ZipFile("data.zip", 'r') as zip_ref:
      zip_ref.extractall("MELA/",pwd=password.encode('utf-8'))

In [2]:
from datasets import concatenate_datasets

MELA_all_languages = ['en', 'zh', 'it', 'ru', 'de', 'fr', 'es', 'ja', 'ar', 'is']
MELA_languages_intersection = ['en', 'ru', 'de', 'es']
datasets_mela_intersection = []
datasets_mela_all_languages = []
mela_train_all =[]
mela_dev_all =[]
mela_test_all =[]


for language_i in MELA_languages_intersection:
    MELA_dataset_i = load_dataset(
        "csv",
        data_files = {
            "train": "MELA/data/v1.0/" + language_i + "/train.csv",
            "validation": "MELA/data/v1.0/" + language_i + "/dev.csv",
            "test": "MELA/data/v1.0/" + language_i + "/test.csv"
        }
    )

    MELA_dataset_i = MELA_dataset_i.select_columns(["sentence","label"])
    MELA_joint = concatenate_datasets([
        MELA_dataset_i["train"],
        MELA_dataset_i["validation"],
        MELA_dataset_i["test"]
    ]).to_pandas()

    datasets_mela_intersection.append(MELA_joint)


for language_i in MELA_all_languages:
    MELA_dataset_i = load_dataset(
        "csv",
        data_files = {
            "train": "MELA/data/v1.0/" + language_i + "/train.csv",
            "validation": "MELA/data/v1.0/" + language_i + "/dev.csv",
            "test": "MELA/data/v1.0/" + language_i + "/test.csv"
        }
    )

    MELA_dataset_i = MELA_dataset_i.select_columns(["sentence","label"])
    MELA_joint = concatenate_datasets([
        MELA_dataset_i["train"],
        MELA_dataset_i["validation"],
        MELA_dataset_i["test"]
    ]).to_pandas()

    mela_train_all.append(MELA_dataset_i["train"].to_pandas())
    mela_dev_all.append(MELA_dataset_i["validation"].to_pandas())
    mela_test_all.append(MELA_dataset_i["test"].to_pandas())

    print(language_i, len(MELA_joint))

    datasets_mela_all_languages.append(MELA_joint)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

...

ar 1017


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

is 2298


In [3]:
mela_train_all = pd.concat(mela_train_all, ignore_index=True)
mela_dev_all = pd.concat(mela_dev_all, ignore_index=True)
mela_test_all = pd.concat(mela_test_all, ignore_index=True)

len(mela_train_all), len(mela_dev_all), len(mela_test_all)

(33293, 6140, 7421)

In [4]:
mela_dev_all["label"].value_counts()

label
1    4741
0    1399
Name: count, dtype: int64

In [5]:
(1.2950854250158195 + 4.388849177984274)

5.683934603000093

In [6]:
(1/(4741/6140))/5.68,  (1/(1399/6140))/5.68

(0.22800799736194005, 0.7726847144338511)

In [7]:
mela_weighs = torch.tensor([0.7726847144338511, 0.22800799736194005])

NameError: name 'torch' is not defined

In [40]:
mela_train_all_X = mela_train_all["sentence"]
mela_train_all_y = mela_train_all["label"]

In [41]:
MELA_intersection = pd.concat(datasets_mela_intersection, ignore_index=True)
MELA_all_languages = pd.concat(datasets_mela_all_languages, ignore_index=True)

In [42]:
len(MELA_intersection), len(MELA_all_languages)

(23228, 46854)

In [43]:
MELA_all_languages["label"].value_counts()

label
1    34977
0    11877
Name: count, dtype: int64

In [44]:
MELA_intersection["label"].value_counts()

label
1    16734
0     6494
Name: count, dtype: int64

In [45]:
MELA_intersection_X = MELA_intersection["sentence"]
MELA_intersection_y = MELA_intersection["label"]

MELA_all_languages_X = MELA_all_languages["sentence"]
MELA_all_languages_y = MELA_all_languages["label"]

## Grammatical

In [46]:
Grammatical_DistilBertLast3LayerFusion_config = DistilBertConfig.from_pretrained(
                                                    "distilbert-base-multilingual-cased",
                                                    num_labels=2,
                                                    dropout=0.1,
                                                    attention_dropout=0.1,
                                                    seq_classif_dropout=0.1,
                                                )

Grammatical_DistilBertLast3LayerFusion_model = DistilBertLast3LayerFusion.from_pretrained(
    "distilbert-base-multilingual-cased",
    config=Grammatical_DistilBertLast3LayerFusion_config,
    ignore_mismatched_sizes=True  # important for custom head
)


Some weights of DistilBertLast3LayerFusion were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['merge_classifier.0.bias', 'merge_classifier.0.weight', 'merge_classifier.3.bias', 'merge_classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inspect architecture

In [47]:
print(Grammatical_DistilBertLast3LayerFusion_model) # See notes above on why ReLU activation does not show.

DistilBertLast3LayerFusion(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

Freeze all DestilBERT. We use only as Feature Extractor.

In [48]:
freeze_all_destilbert(Grammatical_DistilBertLast3LayerFusion_model)

In [49]:
show_trainable_and_frozen(Grammatical_DistilBertLast3LayerFusion_model)

Frozen: distilbert.embeddings.word_embeddings.weight
Frozen: distilbert.embeddings.position_embeddings.weight
Frozen: distilbert.embeddings.LayerNorm.weight
Frozen: distilbert.embeddings.LayerNorm.bias
Frozen: distilbert.transformer.layer.0.attention.q_lin.weight
Frozen: distilbert.transformer.layer.0.attention.q_lin.bias
Frozen: distilbert.transformer.layer.0.attention.k_lin.weight
Frozen: distilbert.transformer.layer.0.attention.k_lin.bias
Frozen: distilbert.transformer.layer.0.attention.v_lin.weight
Frozen: distilbert.transformer.layer.0.attention.v_lin.bias
Frozen: distilbert.transformer.layer.0.attention.out_lin.weight
Frozen: distilbert.transformer.layer.0.attention.out_lin.bias
Frozen: distilbert.transformer.layer.0.sa_layer_norm.weight
Frozen: distilbert.transformer.layer.0.sa_layer_norm.bias
Frozen: distilbert.transformer.layer.0.ffn.lin1.weight
Frozen: distilbert.transformer.layer.0.ffn.lin1.bias
Frozen: distilbert.transformer.layer.0.ffn.lin2.weight
Frozen: distilbert.transf

Train the model

In [83]:
training_args_Grammatical_DistilBertLast3LayerFusion = TrainingArguments(

    output_dir                 = "Saved_models/MELA_FE",
    #eval_strategy              = "epoch",
    save_strategy              = "epoch",
    per_device_eval_batch_size = 32,
    #load_best_model_at_end     = True,

    # Reproducibility
    seed=42,       # Training randomness for weights and dropout.
    data_seed=42,  # Randomness for sampling-shuffling data.

    #Learning rate behaviour
    #learning_rate = default
    warmup_ratio  = 0.1,
    lr_scheduler_type= "linear",

    num_train_epochs            = 4,
    per_device_train_batch_size = 32,

    weight_decay=0.01  # Regularization

    #Other params that we do not use
    #push_to_hub --> upload to HuggingFace
    #gradient_accumulation_steps--> Effective batch size = batch * accumulation_steps

)

Grammatical_DistilBertLast3LayerFusion_trainer = WeightedCrossEntropyTrainer(
    model             = Grammatical_DistilBertLast3LayerFusion_model,
    args              = training_args_Grammatical_DistilBertLast3LayerFusion,
    train_dataset     = dataset_mela_tokenized,
    #eval_dataset      = tokenized_grammar_validation,
    processing_class  = tokenizer,
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,

    class_weights     = mela_weighs,

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [75]:
def tokenize_function_mela(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=512)

dataset_mela = Dataset.from_pandas(mela_train_all.rename(columns={"label": "labels"}))
dataset_mela_tokenized = dataset_mela.map(tokenize_function_mela, batched=True)

Map:   0%|          | 0/33293 [00:00<?, ? examples/s]

In [77]:
mela_train_all.rename(columns={"label": "labels"})

,sentence,labels
0,"Our friends won't buy this analysis, let alone...",1
1,One more pseudo generalization and I'm giving up.,1
2,One more pseudo generalization or I'm giving up.,1
3,"The more we study verbs, the crazier they get.",1
4,Day by day the facts are getting murkier.,1
...,...,...
33288,Það var haldið sig innan dyra út af óveðrinu.,1
33289,Þetta eru allar hinar þínar þrjár nýju kenningar,0
33290,Ég tel hafa verið mýs í baðkerinu.,1
33291,Í þessari ritgerð ætla ég að kynna bókina.,1


In [78]:
training_args_Grammatical_DistilBertLast3LayerFusion

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,
fp1

In [84]:
os.makedirs("Saved_models", exist_ok=True)  # create folder if not existing yet.
Grammatical_DistilBertLast3LayerFusion_model.train()     # Set train mode On

start = time.perf_counter()  # Start point for time Track
Grammatical_DistilBertLast3LayerFusion_trainer.train()              # Train the model
torch.cuda.synchronize()     # Compute and print time spent training
end = time.perf_counter()
elapsed = end - start
print_training_time(elapsed)

Step,Training Loss
500,0.692100
1000,0.680800
1500,0.660800
2000,0.661000
2500,0.649000
3000,0.644500
3500,0.639600
4000,0.632500




________________________________________
Training Time:
0:06:50.511579
[410.51157859998057 Seconds]
________________________________________



In [45]:


#Feature Extraction using frozen base model taking representative vector from 3 last layers.
Grammatical_DistilBertLast3LayerFusion_reload = DistilBertLast3LayerFusion.from_pretrained(
    "Saved_models/MELA_FE/checkpoint-4164"
)

In [46]:
input_text_tokenizer_reloaded = AutoTokenizer.from_pretrained("Saved_models_FT_and_FE/Comprehensible_DBSC_FT_3_layers/checkpoint-3780")
#Any checkpoint ok as all are default DistilBERT Multilingual cased Tokenizer.

The tokenizer you are loading from 'Saved_models_FT_and_FE/Comprehensible_DBSC_FT_3_layers/checkpoint-3780' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


#  .

In [47]:
grammatical_model_for_inference_list = \
    [
     Grammatical_DistilBertLast3LayerFusion_reload,

     ]

grammatical_model_names = \
    [
     "Grammatical_DistilBertLast3LayerFusion_reload"

     ]

In [48]:
#plot_roc_curve(y_true, probabilities)
def compute_predicted_probabilities(X_data,
                               y_true_labels,
                               model_for_inference,
                               input_text_tokenizer,
                               data_loader_batch_size = 32):
    logits_predictions = compute_predictions(X_data, model_for_inference, input_text_tokenizer, data_loader_batch_size)
    softmax_predictions = torch.softmax(logits_predictions, dim=1)  # Softmax for probabilistic interpretation.
    return softmax_predictions[:, 1]

def compute_predictions(X_data, model_for_inference, input_text_tokenizer, data_loader_batch_size = 32):
    X =list(X_data)
    #Collate function and dataloader for parallel inference
    def collate_input(batch):
        return input_text_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)

    # Use Dataloader class passing as input a List of strings.
    # This is Ok as List provides all required operations that a Dataset would provide for building a DataLoader
    loader = DataLoader(X, batch_size=data_loader_batch_size, collate_fn=collate_input)

    # Perform inference using X data
    #switch pytorch from training mode to evaluation-inference mode. Important to disable train only operations.
    model_for_inference.eval()

    #If available save model in GPU
    inference_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_for_inference.to(inference_device)


    all_outputs = []
    with torch.no_grad():
      for inputs in loader:
          inputs = {k: v.to(inference_device) for k, v in inputs.items()} #If available save
                                                                # data in GPU
          logits = model_for_inference(**inputs).logits
          all_outputs.append(logits.detach().cpu())
          del inputs, logits
          torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.empty_cache()
    inference_outputs_tensor = torch.cat(all_outputs, dim=0)


    #switch/restore train mode
    model_for_inference.train()

    return inference_outputs_tensor

In [49]:
grammatical_df_results = compute_predicted_probabilities(
                                     X_data                   = mela_test_all["sentence"],
                                     y_true_labels            = mela_test_all["label"],
                                     model_for_inference = Grammatical_DistilBertLast3LayerFusion_reload,

                                     input_text_tokenizer     = tokenizer)

In [50]:
grammatical_df_results

tensor([0.2005, 0.6067, 0.4264,  ..., 0.5818, 0.5455, 0.3695])

In [51]:
from sklearn.metrics import roc_auc_score, roc_curve

#auc = roc_auc_score(mela_test_all["label"], grammatical_df_results)
#fpr, tpr, thresholds = roc_curve(y_true, y_prob)
#auc

In [52]:
def compute_evaluation_metrics(X_data,
                               y_true_labels,
                               model_for_inference,
                               input_text_tokenizer,
                               data_loader_batch_size = 32,
                               print_report=False,
                               threshold_hp = 0.5):
    logits_predictions = compute_predictions(X_data, model_for_inference, input_text_tokenizer, data_loader_batch_size)
    softmax_predictions = torch.softmax(logits_predictions, dim=1)  # Softmax for probabilistic interpretation.
    return compute_model_metrics(softmax_predictions[:, 1], y_true_labels, print_report=print_report, threshold_hp = threshold_hp)

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def compute_model_metrics(predictions_class_1, y_true, threshold_hp = 0.5, print_report=False):
    """
    :param predictions_class_1: Vector of membership probabilities for Class 1 (that instance belongs to class 1).
    :param y_true:  Vector with correct True labels for each sample in dataset.
    :param threshold_hp: A [0-1] custom [>=] threshhold to assign class 1 membership (Optional with default 0.5).
    :param print_report: If true, prints out all computed metrics including the confusion matrix values. If False, returns a dictionary with name and value of metrics. Useful for metrics post epoch evaluation during training.
    """

    assert (0.0 <= threshold_hp <= 1.0), f"Threshold_hp must be a number in range [0-1]"
    y_pred  = (predictions_class_1 >= threshold_hp).long() # Model predicted classification Class-labels
                                                           # long() casts boolean to Int64 in Tensors


    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    accuracy       = accuracy_score(y_true, y_pred)
    precision      = precision_score(y_true, y_pred)
    recall         = recall_score(y_true, y_pred)  # a.k.a. sensitivity
    specificity    = tn / (tn + fp)
    f1             = f1_score(y_true, y_pred)

    youden_j_statistic   = recall + specificity - 1.0
    fpr, tpr, thresholds = roc_curve(y_true, predictions_class_1)
    roc_auc              = auc(fpr, tpr)

    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html
    # https://en.wikipedia.org/wiki/Pearson_correlation_coefficient
    pearson_correlation_coefficient, p_value = pearsonr(y_true, predictions_class_1)

    if (print_report):
        print("TN =",tn, "FP =",fp, "FN =",fn, "TP =",tp)
        print(f"Accuracy                  : {accuracy:.4f}", " |",tn + tp, "/" ,tn+fp+fn+tp)
        print(f"F1 Score                  : {f1:.4f}")
        print(f"Precision                 : {precision:.4f}", " |",tp, "/" ,tp+fp)
        print(f"Sensitivity (Recall, TPR) : {recall:.4f}", " |", tp, "/" ,tp+fn)
        print(f"Specificity (TNR)         : {specificity:.4f}", " |", tn, "/" ,tn+fp) #
        print(f"Youden's J statistic      : {youden_j_statistic:.4f}")
        print(f"ROC AUC                   : {roc_auc:.4f}")
        print(f"Pearson correlation       : r = {pearson_correlation_coefficient:.4f}", f" p = {p_value}")

        return None
    else:
        results = {
        "Accuracy":accuracy,
        "F1":f1,
        "Precision":precision,
        "Sensitivity (Recall, TPR)":recall,
        "Specificity (TNR)":specificity,
        "Youden's J statistic":youden_j_statistic,
        "ROC AUC":roc_auc,
        "Pearson correlation":pearson_correlation_coefficient
        }
        return results

In [53]:
compute_evaluation_metrics(
                               X_data               =  mela_test_all["sentence"],
                               y_true_labels        =  mela_test_all["label"],
                               model_for_inference  = Grammatical_DistilBertLast3LayerFusion_reload,
                               input_text_tokenizer = tokenizer,
                               data_loader_batch_size = 32,
                               print_report=True)

TN = 1322 FP = 703 FN = 2370 TP = 3026
Accuracy                  : 0.5859  | 4348 / 7421
F1 Score                  : 0.6632
Precision                 : 0.8115  | 3026 / 3729
Sensitivity (Recall, TPR) : 0.5608  | 3026 / 5396
Specificity (TNR)         : 0.6528  | 1322 / 2025
Youden's J statistic      : 0.2136
ROC AUC                   : 0.6468
Pearson correlation       : r = 0.2299  p = 1.4290961789921073e-89


In [55]:
from sklearn.metrics import matthews_corrcoef

y_pred = (grammatical_df_results >= 0.5).long()
mcc = matthews_corrcoef(mela_test_all["label"], y_pred)
mcc

0.19031546393348958